# Explicabilidad y simulador de riesgo por vano

Hermano de `04_uiti_vano_trayectorias_vano.ipynb`: reutiliza su clasificacion KMeans por
vano x ventana -- que nunca se reajusta aqui -- y agrega encima un simulador de "que
pasaria si" a nivel de vano.

**Un solo modelo y una sola unidad.** Todo lo que produce el boton "Simular" -- el mapa
**Criticidad Simulada**, el **top 5 de variables por vano**, el **grafo de relevancia** y
la comparacion **base contra simulado** -- viene del MIL entrenado en
`05_mil_vano_ventana`, que puntua **bolsas**: una bolsa es una celda (vano, ventana) y sus
instancias son los eventos de ese vano en esa ventana. La clase sale de
`asignar_clase(n_obs observado, u-hat predicho)` sobre la geometria KMeans de 04, la misma
del mapa base, de modo que los dos mapas comparten paleta por construccion y no por
convencion. `n_obs` nunca se simula: es un eje del espacio que define la clase.

**Que mide cada mapa.** **Criticidad Original** (izquierda) es el grupo historico que 04
ya calculo sobre eventos observados. **Criticidad Simulada** (derecha) es lo que el modelo
predice al aplicar las variables del simulador. Son dos mediciones distintas y por eso
nunca comparten titulo, aunque si comparten la escala de color, que es la misma por
construccion.

**Se estudian hasta cinco vanos a la vez.** Cada uno recibe su propia columna de controles,
su propia serie de tiempo y su propio top de variables. Ese tope no es decorativo: es lo
que hace que la pregunta del tablero sea "que le pasa a ESTE vano" y no "que le pasa al
promedio de un circuito".

**Requiere dos artefactos de `05_mil_vano_ventana.ipynb`**: `data/models/mil_vano_ventana_v1.pt`
y `data/derived/bolsas_mil_full.joblib`. Los dos viven bajo `data/`, que git ignora; si
faltan, la celda del modelo falla de inmediato nombrando el cuaderno que los produce.

**La interfaz es este cuaderno, y solo este cuaderno.** Todo se controla y se ve en la
figura de seis paneles con los controles de `ipywidgets` encima. Al ejecutar no se escribe
ningun archivo ni se abre ningun navegador. El panel HTML autocontenido que existia antes
se elimino: era una transcripcion completa del modelo a JavaScript -- casi la mitad del
codigo del cuaderno -- que habia que mantener en paralelo con la version de Python y que
se quedaba atras en cada cambio.

In [ ]:
import asyncio
import sys
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError('Este cuaderno requiere ipywidgets para la interfaz interactiva.') from exc
from IPython.display import display

# Sube desde el cwd hasta la raiz del repo (marcada por la carpeta src/), igual que 09.
# Se agregan ROOT y ROOT/src -- no solo src/ -- porque ventanas_015.py importa
# `scripts.extract_geometrias_014` (paquete de nivel de repo, igual que en notebook 10).
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for _path_a_agregar in (ROOT, ROOT / 'src'):
    if str(_path_a_agregar) not in sys.path:
        sys.path.insert(0, str(_path_a_agregar))

# Un kernel que ya importo estos paquetes se queda con la version VIEJA en `sys.modules`:
# "Run All" sin reiniciar NO vuelve a leer el disco. Un rename en src/ -- por ejemplo
# `SelectorVanos._caja` -> `.caja` -- estalla entonces como AttributeError diez celdas mas
# abajo, con el codigo del disco ya correcto. Se purgan ANTES de importarlos, asi el
# cuaderno corre SIEMPRE contra la fuente actual, con o sin reinicio de kernel. Va aqui y no
# como `importlib.reload`: reload no rehace los objetos ya construidos con la clase vieja,
# y este cuaderno los reconstruye todos de esta celda para abajo.
for _modulo in [m for m in list(sys.modules)
                if m.split('.')[0] in ('chec_impacto', 'chec_local_interpreter', 'scripts')]:
    del sys.modules[_modulo]

from chec_impacto.data import procesar_dataset_completo
from chec_impacto.models.criticality_assignment import (
    CLAVE_ESPACIO_CANONICO,
    GEOMETRIAS_SHA1_ESPERADO,
    cargar_geometria_014,
    verificar_sha1_geometrias,
)
from chec_impacto.data.bags import cargar_bolsas
from chec_impacto.models.mil_persistencia import cargar_modelo_mil
from chec_impacto.training import resolve_training_device
from chec_local_interpreter.mil_simulador_015 import (
    gates_de_bolsas,
    grafo_de_gates,
    seleccionar_bolsas,
    sensibilidad_minmax_por_vano,
    simular_bolsas,
    trazas_grafo,
    valores_actuales_por_vano,
)
from chec_local_interpreter.simulador_variables import (
    columnas_panel,
    rotulo_en_barra,
    knobs_bloqueados,
    knobs_simulables,
    tabla_variables_simulables,
)
from chec_local_interpreter.vano_controls import build_knobs, expand_knob_overrides
from chec_local_interpreter.vano_widgets import (
    MAX_VANOS_ANALISIS,
    construir_selector_casillas,
    construir_selector_vanos,
)
from chec_local_interpreter.ventanas_015 import (
    CAMBIOS,
    CAMBIO_EMPEORA,
    CAMBIO_IGUAL,
    CAMBIO_MEJORA,
    bounds_de_fids,
    cajas_por_cambio_de_grupo,
    cajas_seleccion,
    capas_mapa_historico,
    cargar_clases_desde_014,
    centro_y_zoom,
    construir_hist_class_cache,
    construir_mask_cache,
    construir_tabla_vano_ventana,
    construir_ventanas,
    fid_de_punto,
    clases_de_series,
    series_temporal_vanos,
)
from scripts.extract_geometrias_014 import (
    DEFAULT_NOTEBOOK_PATH,
    DEFAULT_OUTPUT_PATH,
    extraer_geometrias_014,
)

# Sonda del contrato que rompio el cuaderno dos veces. Si el kernel siguiera sirviendo una
# version vieja de vano_widgets, falla AQUI -- primera celda, mensaje que dice que hacer --
# en vez de a los 10 minutos de procesamiento, en la celda del panel.
_sonda = construir_selector_vanos(['0'])
assert hasattr(_sonda, 'caja'), (
    'vano_widgets viejo en memoria: el selector de casillas sin `.caja`. '
    'Reinicia el kernel. '
    f'(modulo cargado desde {sys.modules["chec_local_interpreter.vano_widgets"].__file__})'
)
del _sonda

In [ ]:
# Ventana climatica igual que 03_mgcecdl_training / 09_simulador: cambiarla generaria un
# set de features distinto al que el modelo cargado en la celda SEAM espera.
VENTANA_CLIMATICA_HORAS = 12
# El espacio de agrupamiento no se elige: viene fijo de `criticality_assignment.py`.
# La clave paso de '2' a '0' el 2026-08-09, cuando 04 dejo de enumerar ocho espacios;
# la geometria es la MISMA, solo cambio el indice bajo el que esta archivada, asi que
# esto se lee de la constante y no se escribe a mano.
CLAVE_ESPACIO = CLAVE_ESPACIO_CANONICO
DEVICE = resolve_training_device('auto')

# Misma paleta que 01.4: los grupos historicos de este cuaderno SON los de 01.4, nunca se
# reajustan, asi que el color tiene que significar lo mismo en los dos cuadernos.
NOMBRES_GRUPOS = ['Bajo', 'Medio', 'Medio-Alto', 'Alto']
COLORES_GRUPOS = ['rgb(252,187,161)', 'rgb(251,106,74)', 'rgb(203,24,29)', 'rgb(103,0,13)']
# UN solo codigo de ausencia en los DOS mapas: negro, el `COLOR_SIN_EVENTO` de 01.4.
# El vano sin eventos en la ventana no tiene clase, y la ausencia no es la clase mas baja;
# que el base lo pintara gris y el simulado negro obligaba a recordar dos codigos para la
# misma cosa. Los cuatro colores KMeans significan lo mismo en los dos mapas.
COLOR_SIN_EVENTO = 'rgb(0,0,0)'
COLOR_MARCADO = '#0072b2'
# Equipos: mismos colores que 01.4, por el mismo motivo que la paleta de grupos --
# un naranja tiene que seguir siendo un transformador al pasar de un cuaderno a otro.
COLOR_TRAFO = '#f59e0b'
COLOR_SWITCH = '#7c3aed'
ANCHO_MAPA = 3.0
ANCHO_MAPA_MARCADO = round(ANCHO_MAPA * 1.4, 2)

# Fila 1, paridad 01.4: un vano MARCADO se dibuja con el color de SU clase, sobre un halo
# blanco que lo despega del fondo (01.4: `width=ANCHO_MAPA_RESALTE * 2.6, color='white'`).
# Un color plano de "seleccionado" encima de la clase congela lo que se ve: la ventana
# cambia la clase por debajo y el vano marcado sigue igual en pantalla. COLOR_MARCADO
# queda solo para la fila 2, donde la clase la pone el modelo y no el KMeans.
COLOR_HALO = 'white'
ANCHO_HALO = round(ANCHO_MAPA_MARCADO * 2.6, 2)
# El vano SELECCIONADO se encierra ademas en una caja amarilla translucida: su caja
# envolvente, el rectangulo min/max de sus coordenadas. El halo blanco y el ancho extra
# lo separan de su vecino inmediato, pero no lo hacen ENCONTRABLE en un circuito de
# cientos de tramos; una caja si, y sigue siendo una caja a cualquier zoom, donde una
# linea deja de distinguirse de las de al lado.
# Va por `layout.map.layers` con `below='traces'` y NO como traza. Las dos cosas importan:
# una traza rellena por encima se comeria el clic -- que es justo lo que alterna la
# seleccion -- y ademas tiniria de amarillo la linea del vano, borrando el color de su
# clase. Debajo de las trazas, la caja rodea al vano y la clase se sigue leyendo.
COLOR_CAJA_SELECCION = '#ffd400'
OPACIDAD_CAJA_SELECCION = 0.5
# El recuadro del mapa SIMULADO no contesta "cual estoy estudiando" -- eso ya lo dice el
# del mapa base, con el mismo vano encerrado a la izquierda -- sino QUE LE PASO al vano:
# verde claro si bajo de grupo de criticidad, amarillo si se quedo en el mismo, rojo si
# subio. Son TRES capas y no una porque una capa de `layout.map.layers` pinta con UN color.
# El amarillo es exactamente el del mapa base a proposito: "no cambio" es justo el estado
# en que los dos mapas dicen lo mismo, y un cuarto color inventaria una diferencia que no
# hay.
COLOR_CAJA_MEJORA = '#4ade80'
COLOR_CAJA_IGUAL = COLOR_CAJA_SELECCION
COLOR_CAJA_EMPEORA = '#dc2626'
# Lado minimo de la caja, en grados (~50 m a esta latitud). Un vano exactamente norte-sur
# tiene caja envolvente de ancho CERO, y cero pixeles de ancho no se ve.
LADO_MINIMO_CAJA = 0.00045
# Margen a cada lado (~10 m): sin el, el borde de la caja cae encima del trazo del vano y
# no se distingue cual es cual.
MARGEN_CAJA = 0.00009
OPACIDAD_NUBE = 0.45               # 01.4, para que la nube de fondo no tape el resaltado
OPACIDAD_FRONTERA = 0.28           # 01.4: el contorno es fondo, no dato
# Paleta de 01.4 para las series por vano: apta para daltonismo y distinta de la escala de
# grupos, porque aqui el color identifica AL VANO, no a su clase.
COLORES_VANOS = ['#0072b2', '#009e73', '#cc79a7', '#56b4e9', '#e69f00', '#8c564b']
# Cuantas barras lleva cada grupo del top por vano. DIEZ y no cinco: el barrido puntua
# trece variables numericas y cortar en cinco dejaba fuera de la vista mas de la mitad del
# ranking que ya se calculo -- las pasadas del modelo son las mismas, solo cambia cuantas
# se muestran. Lo que el cinco protegia era el rotulo escrito dentro de la barra, y de eso
# se encarga ahora la cascada resumen -> inicial -> nada de
# `simulador_variables.rotulo_en_barra`, que decide barra por barra segun lo que mida.
TOP_VARIABLES_POR_VANO = 10
# Fuente del rotulo dentro de la barra. Los anchos de caracter con los que
# `rotulo_en_barra` decide si el nombre cabe estan MEDIDOS a este tamanio
# (`simulador_variables.TAM_FUENTE_MEDIDO`): cambiarlo aqui sin volver a medir alla deja
# al panel decidiendo con numeros de otra fuente.
TAM_FUENTE_BARRA = 8
# El punto de la VENTANA VIGENTE en las dos series se dibuja al triple, como el dia
# vigente en la serie del cuaderno 01. `marker.size` es un ARRAY por eso: mover el
# deslizador solo reescribe ese arreglo y el punto grande viaja con el.
SERIE_TAM_UITI = 9
SERIE_TAM_EVENTOS = 8
FACTOR_PUNTO_ACTIVO = 3
# Los dos violines de la fila 4: la MISMA cantidad medida antes y despues de simular.
# Colores fuera de la escala de criticidad a proposito -- aqui el color distingue dos
# CORRIDAS, no dos niveles, y reusar la paleta de grupos invitaria a leerlos como clases.
# El punto de la serie sin celda en la ventana: no tiene grupo, y eso NO es el grupo mas
# bajo -- es la ausencia del dato. Gris, fuera de la escala de criticidad.
COLOR_SIN_GRUPO = '#94a3b8'
COLOR_VIOLIN_BASE = '#94a3b8'
COLOR_VIOLIN_SIMULADO = '#0072b2'
# Paleta de los MODOS de variable en el grafo. Deliberadamente fuera de la familia de los
# grupos KMeans (rojos/naranjas) y de los equipos: un rojo en el mapa y un rojo en el grafo
# significarian cosas sin ninguna relacion. Se recorre en el orden de las modalidades del
# artefacto.
PALETA_MODALIDADES = ['#0d9488', '#be185d']   # verde azulado y rosa oscuro

# Guion horizontal negro en cada extremo de CADA vano, tenga o no eventos, igual que el
# mapa de 01: grados de longitud a cada lado del extremo (~14 m a esta latitud). Marca
# donde empieza y donde termina un vano, que es lo unico que distingue dos vanos vecinos
# dibujados con el mismo color.
MARCA_VANO = 0.00013
# Densificacion del hover del mapa: el hover de una traza de lineas en Scattermap se
# resuelve contra los VERTICES y no contra la linea, y los tramos de MVLINSEC traen
# exactamente dos. Sin esto el centro de un vano largo no muestra etiqueta y, como Plotly
# solo convierte un clic en evento donde hay hover, tampoco se puede marcar tocandolo ahi.
PASO_VERTICE = 0.00022      # grados ~= 25 m a esta latitud



In [ ]:
# --- Reutilizacion de la geometria KMeans de 01.4 (design section F) -------
# Falla RAPIDO aqui, antes de procesar el dataset completo (celda siguiente): si 01.4 fue
# editado y sus centroides se movieron, no tiene sentido esperar el procesamiento pesado
# para enterarse. `cargar_clases_desde_014` (celda 7, via hist_class_cache) repite esta
# misma verificacion por cada ventana consultada -- barata, y evita que una geometria
# cacheada quede sin recomprobar dentro de la misma sesion.
GEOMETRIAS_PATH = DEFAULT_OUTPUT_PATH
if not GEOMETRIAS_PATH.exists():
    extraer_geometrias_014(DEFAULT_NOTEBOOK_PATH, GEOMETRIAS_PATH)
_sha1_real, _coincide = verificar_sha1_geometrias(GEOMETRIAS_PATH, esperado=GEOMETRIAS_SHA1_ESPERADO)
assert _coincide, (
    f'La geometria KMeans extraida de 01.4 no coincide con la esperada '
    f'(esperado={GEOMETRIAS_SHA1_ESPERADO}, real={_sha1_real}). 01.4 fue modificado; '
    f'01.5 depende de esa geometria.'
)
# La geometria en si (no solo su sha1): la nube KMeans de la fila 3 tiene que dibujarse en
# el MISMO espacio en que se asignan las clases -- el canonico '2' es (log_x=False,
# log_y=True). Leerlo de la geometria y no fijarlo a mano evita que un cambio de espacio
# deje la nube en ejes que ya no corresponden a las fronteras.
GEOMETRIA_014 = cargar_geometria_014(GEOMETRIAS_PATH, CLAVE_ESPACIO)
print(f'Geometria 01.4 verificada -- sha1 coincide ({_sha1_real[:12]}...) | '
      f'espacio {CLAVE_ESPACIO}: log_x={GEOMETRIA_014.logs[0]}, log_y={GEOMETRIA_014.logs[1]}')

In [ ]:
DATA_PATH = ROOT / 'data' / 'Indicadores_vano_v3.csv'
VARIABLES_SELECCION_PATH = ROOT / 'data' / 'Variables_seleccion.xlsx'
MODEL_DIR = ROOT / 'data' / 'models'

# Mismo preprocesamiento real usado en entrenamiento (03_mgcecdl_training / 09_simulador):
# sin muestreo ni filtro de UITI, para que context_df quede alineado FILA A FILA con X --
# la clave que permite reusar la MISMA mascara (circuito, ventana) para el mapa historico
# (sin modelo) y, en un PR futuro, para las predicciones del modelo sobre esas mismas filas.
datos = procesar_dataset_completo(
    path_clima=DATA_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target='UITI_VANO',
    filtro_uiti_max=None,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

feature_names = list(datos['features'])
X_raw_model = np.asarray(datos['X'], dtype=np.float32)
Xdf = datos['Xdata'].copy().reset_index(drop=True)
context_df = datos['df_original_copy'].copy().reset_index(drop=True)
label_encoders = datos.get('label_encoders', {})
max_values_imputed = datos.get('max_values_imputed', {})

# Ya no se construye el escalador min-max de MGCECDL. El simulador y la importancia de
# variables corren sobre el modelo MIL del cuaderno 05, cuya matriz de instancias es RAW,
# asi que `preparar_splits_estratificados` + `escalar_features_minmax_mgcecdl` -- de lo
# mas caro de esta celda -- no alimentaban ya a nadie.
assert len(context_df) == len(X_raw_model), (
    'context_df y X_raw_model deben quedar alineados fila a fila'
)
print(f'{len(context_df):,} filas | {len(feature_names)} features')

In [ ]:
# --- UN solo modelo: el MIL por bolsas del cuaderno 05 (cierra el SEAM D1) ------------
# El tablero entero -- mapa "Criticidad Simulada", grafo reconstruido e "Importancia
# Variables" -- responde a este modelo y a esta unidad: la BOLSA (vano x ventana), que es
# la unidad en la que 04 define la criticidad. MGCECDL por fila salio del cuaderno: tener
# dos modelos contestando paneles vecinos del mismo tablero significaba que el panel y el
# mapa hablaban de cosas distintas sin que nada en pantalla lo dijera.
# Requiere DOS artefactos que produce el cuaderno 05 y que viven bajo `data/` (ignorado
# por git): el modelo y el cache de bolsas. Falla AQUI, con el nombre del cuaderno que los
# genera, en vez de a los diez minutos en la celda del boton.
RUTA_MODELO_MIL = MODEL_DIR / 'mil_vano_ventana_v1.pt'
RUTA_BOLSAS_MIL = ROOT / 'data' / 'derived' / 'bolsas_mil_full.joblib'
for _ruta in (RUTA_MODELO_MIL, RUTA_BOLSAS_MIL):
    assert _ruta.exists(), (
        f'Falta {_ruta.name}: lo produce 05_mil_vano_ventana.ipynb. Corre ese cuaderno '
        'antes que este.'
    )

BOLSAS = cargar_bolsas(RUTA_BOLSAS_MIL)
X_INST, FEATURES_MIL, BAG_INDEX = BOLSAS['X'], BOLSAS['features'], BOLSAS['bag_index']
# `device='cpu'` a proposito y no DEVICE: una seleccion son decenas de instancias, asi que
# el traslado a GPU/MPS cuesta mas de lo que ahorra, y saca una variable de dtype de en
# medio de un camino interactivo.
MIL = cargar_modelo_mil(RUTA_MODELO_MIL, device='cpu', features_esperadas=FEATURES_MIL)

# La guarda que hace comparables los dos mapas: si el MIL se hubiera entrenado con OTRA
# geometria KMeans, sus clases usarian los mismos 4 colores para significar otra cosa.
for _campo in ('offset', 'scale', 'centroides'):
    assert np.allclose(getattr(MIL.geometria, _campo), getattr(GEOMETRIA_014, _campo)), (
        f'La geometria del modelo MIL difiere de la de 01.4 en {_campo}: sus clases NO son '
        'las del mapa base y no pueden compartir la paleta.'
    )
assert tuple(MIL.geometria.logs) == tuple(GEOMETRIA_014.logs)

# Las 70 primeras features del MIL son exactamente las de MGCECDL (22 estaticas + 48 de
# clima); las 10 restantes son COD_CAUSA y sus indicadores, que no son controles del
# simulador. Por eso el catalogo de knobs de la celda siguiente sirve para los dos.
assert list(FEATURES_MIL[:len(feature_names)]) == list(feature_names), (
    'Las features del MIL ya no empiezan por las de MGCECDL: el catalogo de knobs '
    'apuntaria a columnas equivocadas.'
)
# Los MODOS de variable con los que el modelo agrupa las columnas -- son los mismos que
# usa la fusion FiLM (el clima reescala lo estructural), asi que colorear los nodos del
# grafo por modalidad muestra exactamente la particion que el modelo usa por dentro.
COLUMNAS_MODALIDAD = {m: set(int(i) for i in idx)
                      for m, idx in MIL.model.base.modality_feature_indices.items()}
MODALIDADES_MIL = list(COLUMNAS_MODALIDAD)
assert len(MODALIDADES_MIL) == len(PALETA_MODALIDADES), (
    f'El artefacto trae {len(MODALIDADES_MIL)} modalidades y la figura tiene trazas de '
    f'nodo para {len(PALETA_MODALIDADES)}: agrega la traza que falta antes de seguir.'
)
COLORES_MODALIDAD = dict(zip(MODALIDADES_MIL, PALETA_MODALIDADES))

print(f'MIL cargado -- {len(BAG_INDEX.keys):,} bolsas | {X_INST.shape[0]:,} instancias x '
      f'{len(FEATURES_MIL)} features | geometria identica a 01.4')
print('modos de variable: ' + ' | '.join(
    f'{m} ({len(COLUMNAS_MODALIDAD[m])})' for m in MODALIDADES_MIL))

In [ ]:
# --- construir_ventanas + per-(vano, ventana) events + caches (design section A) -------
VENTANAS = construir_ventanas(context_df['FECHA'])
TABLA = construir_tabla_vano_ventana(context_df, VENTANAS)
mask_para = construir_mask_cache(TABLA)
clases_para = construir_hist_class_cache(TABLA, mask_para)

# Aqui se clasificaban las 111 mil celdas de una sola pasada y se submuestreaba una nube
# de 20 mil puntos para el panel KMeans. Ese panel ya no existe, asi que el calculo
# tampoco: eran una pasada de `cargar_clases_desde_014` sobre la tabla entera y ~1,2 MB de
# coordenadas que nadie iba a dibujar. La clase que el mapa necesita la sigue dando
# `clases_para`, ventana por ventana y cacheada.

CIRCUITOS = sorted(TABLA['CIRCUITO'].astype(str).unique())
VANOS_POR_CIRCUITO = {
    c: sorted(g['FID_VANO'].unique().tolist())
    for c, g in TABLA.groupby(TABLA['CIRCUITO'].astype(str))
}

print(f'{len(TABLA):,} celdas vano x ventana con eventos | {len(VENTANAS)} ventanas | '
      f'{TABLA["FID_VANO"].nunique():,} vanos distintos | {len(CIRCUITOS)} circuitos')


# Geometria FISICA de cada vano (no confundir con la geometria KMeans de la celda 4): mismo
# shapefile y mismo join que el mapa de 01.3/01.4. No se extrae a src/ porque es solo
# lectura + reindexado geoespacial, sin logica propia que valga la pena testear por fuera
# de lo que TABLA/capas_mapa_historico ya cubren.
def _norm_id(serie):
    return (serie.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
            .replace({'': pd.NA, '<NA>': pd.NA, 'nan': pd.NA, 'None': pd.NA}))


_lineas = gpd.read_file(ROOT / 'data' / 'GEO' / 'MVLINSEC.shp')
if str(_lineas.crs) != 'EPSG:4326':
    _lineas = _lineas.to_crs('EPSG:4326')
_lineas['FID_VANO_GEO'] = _norm_id(_lineas['G3E_FID'])
_utiles = _lineas[_lineas['CIRCUITO'].astype(str).isin(set(CIRCUITOS))]

GEO_POR_CIRCUITO = {}
for _c, _g in _utiles.groupby(_utiles['CIRCUITO'].astype(str)):
    fids, lats, lons = [], [], []
    for _fid, _geom in zip(_g['FID_VANO_GEO'], _g.geometry):
        if _geom is None or _geom.is_empty:
            continue
        for _p in ([_geom] if _geom.geom_type == 'LineString' else list(getattr(_geom, 'geoms', []))):
            xs, ys = _p.xy
            fids.append(str(_fid))
            lats.append([round(v, 5) for v in ys])
            lons.append([round(v, 5) for v in xs])
    if fids:
        # `bounds` es lo que permite encuadrar el mapa sobre el circuito elegido, igual
        # que 01.4: [lat_min, lat_max, lon_min, lon_max].
        _la = [v for l in lats for v in l]
        _lo = [v for l in lons for v in l]
        GEO_POR_CIRCUITO[_c] = {
            'fids': fids, 'lat': lats, 'lon': lons,
            'bounds': [round(min(_la), 5), round(max(_la), 5),
                       round(min(_lo), 5), round(max(_lo), 5)],
        }


def _equipo(nombre):
    """Transformadores e interruptores del circuito, igual que 01.4 celda 5. Si el
    shapefile no esta, el mapa se dibuja sin equipos en vez de fallar: son contexto
    de lectura, no el dato del tablero."""
    ruta = ROOT / 'data' / 'GEO' / nombre
    if not ruta.exists():
        return {}
    g = gpd.read_file(ruta)
    if str(g.crs) != 'EPSG:4326':
        g = g.to_crs('EPSG:4326')
    g = g[g['CIRCUITO'].astype(str).isin(set(CIRCUITOS))]
    g = g[g.geometry.notna() & ~g.geometry.is_empty]
    return {c: {'lat': [round(float(p.y), 5) for p in gg.geometry],
                'lon': [round(float(p.x), 5) for p in gg.geometry]}
            for c, gg in g.groupby(g['CIRCUITO'].astype(str))}


TRAFOS = _equipo('GDBCHEC_TRANSFOR.shp')
SWITCHES = _equipo('SWITCHES.shp')

# UITI y eventos por vano y ventana: solo alimentan el hover, igual que 01.4. El grupo
# NO se guarda aqui -- sale de `clases_para`, que es la unica fuente de clases.
DATOS_VENTANA = [{} for _ in VENTANAS]
for _fid, _vi, _u, _n in zip(TABLA['FID_VANO'], TABLA['ventana_i'],
                             TABLA['uiti_acumulado'], TABLA['num_eventos']):
    DATOS_VENTANA[int(_vi)][str(_fid)] = (float(_u), int(_n))

print(f'{len(GEO_POR_CIRCUITO)} circuitos con geometria fisica | '
      f'{sum(len(v["lat"]) for v in TRAFOS.values()):,} transformadores | '
      f'{sum(len(v["lat"]) for v in SWITCHES.values()):,} switches')

In [ ]:
KNOBS = build_knobs(
    feature_names=feature_names,
    original_feature_df=Xdf,
    label_encoders=label_encoders,
    max_values_imputed=max_values_imputed,
)
print(f'{len(KNOBS)} controles (Knob catalog, PR2a) -- '
      f'{sum(1 for k in KNOBS if k.kind == "categorical")} categoricos, '
      f'{sum(1 for k in KNOBS if k.kind == "numeric")} numericos, '
      f'{sum(1 for k in KNOBS if k.kind == "constant")} constantes')

In [ ]:
# --- Que variables tiene sentido simular, y cuales no ---------------------------------
# El catalogo de knobs se construye desde la lista de features y no sabe que significa
# ninguna: ofrece el nivel de riesgo por vegetacion -- que se baja con la cuadrilla de la
# semana entrante -- al lado de las coordenadas del vano, como si mover un vano fuera una
# opcion de mantenimiento. Esa distincion vivia solo en la cabeza de quien lee.
# Aqui queda como dato: rango real de cada control y veredicto sobre si simularlo
# significa algo, con el motivo pegado para poder discutirlo. Los veredictos salen del
# diccionario del propio proyecto (`data/Variables_seleccion.xlsx`), citado en cada
# motivo, y viven en `simulador_variables.JUICIO_SIMULACION`, con sus pruebas.
# Una variable nueva del modelo aparece como "Sin evaluar" y no como una palanca mas:
# suponerla valida meteria un control sin revisar en el panel.
TABLA_VARIABLES = tabla_variables_simulables(KNOBS)

_COLOR_VEREDICTO = {
    'Si -- intervencion': '#dcfce7',   # verde: hay una obra detras
    'Si -- escenario': '#dbeafe',      # azul: nadie lo controla, pero es el what-if
    'Limitado': '#fef3c7',             # ambar: una sola lectura lo hace interpretable
    'No': '#fee2e2',                   # rojo: circular o identidad del vano
    'Sin evaluar': '#e5e7eb',
}
_conteo = TABLA_VARIABLES['Sentido de simular'].value_counts()
print(f'{len(TABLA_VARIABLES)} controles simulables -- '
      + ', '.join(f'{_conteo.get(v, 0)} {v}' for v in _COLOR_VEREDICTO if _conteo.get(v, 0)))
print('Las constantes quedan fuera: un control con un solo valor observado no mueve nada.')
_sin_unidad = int((TABLA_VARIABLES['Unidad'] == '').sum())
print(f'{_sin_unidad} sin unidad: categoricas, binarias, indices y las que el diccionario '
      'del proyecto no documenta (ver `simulador_variables.UNIDADES`).')

# `Por que` es una frase entera: sin `white-space: normal` pandas la muestra en una sola
# linea y la tabla se sale de la celda. El ancho se fija por columna para que el motivo
# se lleve el espacio y el rango no.
display(
    TABLA_VARIABLES.style
    .hide(axis='index')
    .format({'vmin': '{:,.4g}', 'vmax': '{:,.4g}'}, na_rep='--')
    .map(lambda v: f'background-color: {_COLOR_VEREDICTO.get(v, "")}',
         subset=['Sentido de simular'])
    .set_properties(**{'white-space': 'normal', 'vertical-align': 'top',
                       'font-size': '12px'})
    .set_properties(subset=['Por que'], **{'width': '460px', 'color': '#4b5563'})
    .set_properties(subset=['Variable'], **{'font-weight': '600'})
    .set_properties(subset=['Unidad'], **{'white-space': 'nowrap'})
    .set_table_styles([{'selector': 'th',
                        'props': [('font-size', '12px'), ('text-align', 'left')]}])
)


In [ ]:
# --- Inventario de trazas -------------------------------------------------------------
# La figura tiene SEIS paneles y no ocho. Los dos mapas ocupan el cuadrante superior, uno
# al lado del otro, y las filas 3 y 4 responden cuatro preguntas distintas sobre los vanos
# elegidos: como vienen en el tiempo, que variable mueve a CADA uno, como se relacionan
# esas variables entre si, y cuanto cambia el UITI al simular.
# Salieron la nube KMeans, la barra de importancia agregada de la seleccion -- la
# reemplaza el top 5 POR VANO, que es la pregunta que sostiene una orden de trabajo -- y
# el violin de eventos por grupo, cuyo hueco pasa a comparar base contra simulado.
#
# El inventario ya no esta congelado por indices sueltos: se ARMA sobre la marcha y `IDX`
# se llena con lo que devuelve `add_trace`. Congelarlo a mano tenia sentido cuando las
# trazas se agregaban de a una en PRs sucesivos; ahora el orden lo fija este bloque y
# cualquier reordenamiento se ve aqui mismo.
IDX = {}

# La caja amarilla del vano seleccionado vive en el LAYOUT del mapa base y no en el
# inventario de trazas: `below='traces'` la deja debajo de todos los tramos, con lo que no
# intercepta ni el hover ni el clic -- que es justo lo que alterna la seleccion -- y no
# tapa el color de clase del vano que esta senialando. Nace vacia; el repintado del mapa
# le escribe el `source`.
CAPA_CAJA_SELECCION = dict(
    sourcetype='geojson', type='fill', below='traces',
    source={'type': 'FeatureCollection', 'features': []},
    color=COLOR_CAJA_SELECCION, opacity=OPACIDAD_CAJA_SELECCION,
)
# El mapa simulado lleva TRES capas de caja, una por desenlace, por la misma razon por la
# que la de la izquierda es una sola: una capa pinta con UN color. Nacen vacias y siempre
# en el mismo orden (`CAMBIOS`), asi que el repintado es una escritura de `source` por capa
# y nunca un quitar y poner capas del mapa -- que en MapLibre reordena lo que hay debajo.
COLOR_POR_CAMBIO = {CAMBIO_MEJORA: COLOR_CAJA_MEJORA,
                    CAMBIO_IGUAL: COLOR_CAJA_IGUAL,
                    CAMBIO_EMPEORA: COLOR_CAJA_EMPEORA}
CAPAS_CAJA_SIMULADA = [
    dict(sourcetype='geojson', type='fill', below='traces',
         source={'type': 'FeatureCollection', 'features': []},
         color=COLOR_POR_CAMBIO[_cambio], opacity=OPACIDAD_CAJA_SELECCION)
    for _cambio in CAMBIOS
]
IDX_CAPA_CAMBIO = {_cambio: _i for _i, _cambio in enumerate(CAMBIOS)}


def _agregar(traza, fila, columna, **kwargs):
    """Agrega la traza y devuelve su indice, que es lo unico que el resto del cuaderno
    necesita saber de ella."""
    _fig.add_trace(traza, row=fila, col=columna, **kwargs)
    return len(_fig.data) - 1


_fig = make_subplots(
    rows=4, cols=4,
    specs=[[{'type': 'map', 'rowspan': 2, 'colspan': 2}, None,
            {'type': 'map', 'rowspan': 2, 'colspan': 2}, None],
           [None, None, None, None],
           [{'type': 'xy', 'colspan': 2, 'secondary_y': True}, None,
            {'type': 'xy', 'colspan': 2}, None],
           [{'type': 'xy', 'colspan': 2}, None, {'type': 'xy', 'colspan': 2}, None]],
    subplot_titles=(
        'Criticidad Original',
        'Criticidad Simulada',
        'UITI acumulado y eventos por ventana',
        f'Top {TOP_VARIABLES_POR_VANO} de variables por vano',
        'Grafo de relevancia',
        'UITI de la seleccion: base contra simulado',
    ),
    row_heights=[0.22, 0.22, 0.28, 0.28],
    # 0,05 y no 0,035: en el hueco entre la serie de tiempo y el top 5 tienen que caber
    # CUATRO cosas -- las marcas del eje secundario de la serie, su rotulo "Eventos", el
    # rotulo "Relevancia variables" del top y las marcas de ese eje. Medido a 1.900 px,
    # con 0,035 el hueco era de 64 px y las marcas del eje secundario ya llegaban a 21 de
    # esos: los dos rotulos se encimaban.
    horizontal_spacing=0.055, vertical_spacing=0.095,
)

# --- Mapa base (filas 1-2, columnas 1-2) ---------------------------------------------
IDX['clases'] = [
    _agregar(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase], legendgroup='hist',
        line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), 1, 1) for _clase in range(4)
]
IDX['sin_dato'] = _agregar(go.Scattermap(
    lat=[], lon=[], mode='lines', name='Sin evento en la ventana', legendgroup='hist',
    line=dict(width=ANCHO_MAPA, color=COLOR_SIN_EVENTO), hovertext=[], hoverinfo='text',
), 1, 1)
# El marcado va en DOS capas, como en 01.4: primero el halo blanco ancho y despues la
# linea con el color de SU clase. Un color plano de "seleccionado" encima congelaria lo
# que se ve: la ventana cambia la clase por debajo y el vano seguiria igual en pantalla.
IDX['marcados'] = _agregar(go.Scattermap(
    lat=[], lon=[], mode='lines', name='Vano marcado', legendgroup='hist', showlegend=False,
    line=dict(width=ANCHO_HALO, color=COLOR_HALO), hovertext=[], hoverinfo='text',
), 1, 1)
IDX['marcados_clases'] = [
    _agregar(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase], legendgroup='hist',
        showlegend=False, line=dict(width=ANCHO_MAPA_MARCADO, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), 1, 1) for _clase in range(4)
]
IDX['marcados_sin_dato'] = _agregar(go.Scattermap(
    lat=[], lon=[], mode='lines', name='Marcado sin eventos', legendgroup='hist',
    showlegend=False, line=dict(width=ANCHO_MAPA_MARCADO, color=COLOR_SIN_EVENTO),
    hovertext=[], hoverinfo='text',
), 1, 1)

# --- Mapa simulado (filas 1-2, columnas 3-4) -----------------------------------------
IDX['pred_clases'] = [
    _agregar(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase], legendgroup='pred',
        showlegend=False, line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), 1, 3) for _clase in range(4)
]
IDX['pred_sin_dato'] = _agregar(go.Scattermap(
    lat=[], lon=[], mode='lines', name='Sin evento / no simulado', legendgroup='pred',
    showlegend=False, line=dict(width=ANCHO_MAPA, color=COLOR_SIN_EVENTO),
    hovertext=[], hoverinfo='text',
), 1, 3)

# Los equipos van DESPUES de los tramos para dibujarse encima. Se repiten por mapa porque
# una traza pertenece a un solo subplot: sin ellos la derecha se leeria como otra geografia.
for _columna_mapa, _leyenda in ((1, True), (3, False)):
    _claves = ('trafos', 'switches') if _columna_mapa == 1 else ('pred_trafos', 'pred_switches')
    for _clave, _nombre, _color, _tam in zip(
            _claves, ('Transformadores', 'Switches'), (COLOR_TRAFO, COLOR_SWITCH), (6, 5)):
        IDX[_clave] = _agregar(go.Scattermap(
            lat=[], lon=[], mode='markers', name=_nombre, legendgroup='equipos',
            showlegend=_leyenda, marker=dict(size=_tam, color=_color),
            hovertext=[], hoverinfo='text',
        ), 1, _columna_mapa)

# --- Fila 3, columnas 1-2: la serie de tiempo de los vanos elegidos --------------------
# Paridad con 03 y 04: la LINEA y el ANILLO del punto llevan el color de identidad del
# vano -- dicen de QUE vano es la serie -- y el RELLENO del punto lleva el color del grupo
# de riesgo en que cayo ese vano en esa ventana. Son dos codigos sobre el mismo dato,
# separados por canal para que los dos se lean a la vez. Un relleno gris es una ventana sin
# celda, que no tiene grupo -- distinto de caer en el mas bajo. Doble eje porque UITI y eventos viven en escalas muy distintas y
# compartir eje aplastaria una de las dos.
# `marker.size` es un ARRAY: el punto de la ventana activa va al triple y viaja con el
# deslizador sin partir la serie en una segunda traza.
_VACIO = [None] * len(VENTANAS)
# El eje x son los INDICES de ventana, pero las marcas llevan la FECHA en que empieza
# cada una y no su etiqueta "V1", "V2": un rotulo "V7" obliga a ir a buscar a que periodo
# corresponde cada vez que se mira el panel. El ano se recorta como en 03 y 04 -- todas
# las ventanas caen en el mismo -- para que las once quepan sin apilarse.
_X_VENTANAS = [v['i'] for v in VENTANAS]
_FECHAS_VENTANA = [str(v['desde'].date()) for v in VENTANAS]
_ANIOS_VENTANA = sorted({f[:4] for f in _FECHAS_VENTANA})
_TICKS_VENTANA = ([f[5:] for f in _FECHAS_VENTANA] if len(_ANIOS_VENTANA) == 1
                  else _FECHAS_VENTANA)
IDX['serie_uiti'] = [
    _agregar(go.Scatter(
        x=_X_VENTANAS, y=list(_VACIO), mode='lines+markers', name='', showlegend=False,
        line=dict(color=COLORES_VANOS[_s], width=2),
        marker=dict(size=[SERIE_TAM_UITI] * len(VENTANAS),
                    color=[COLOR_SIN_GRUPO] * len(VENTANAS),
                    line=dict(width=1.2, color=COLORES_VANOS[_s])),
        hovertext=[], hoverinfo='text', connectgaps=False,
    ), 3, 1, secondary_y=False) for _s in range(MAX_VANOS_ANALISIS)
]
IDX['serie_eventos'] = [
    _agregar(go.Scatter(
        x=_X_VENTANAS, y=list(_VACIO), mode='lines+markers', name='', showlegend=False,
        line=dict(color=COLORES_VANOS[_s], width=1.1, dash='dot'),
        marker=dict(size=[SERIE_TAM_EVENTOS] * len(VENTANAS), symbol='square',
                    color=[COLOR_SIN_GRUPO] * len(VENTANAS),
                    line=dict(width=1.1, color=COLORES_VANOS[_s])),
        hovertext=[], hoverinfo='text', connectgaps=False,
    ), 3, 1, secondary_y=True) for _s in range(MAX_VANOS_ANALISIS)
]
_fig.update_xaxes(title_text=('Inicio de la ventana'
                              + (f' ({_ANIOS_VENTANA[0]})' if len(_ANIOS_VENTANA) == 1 else '')),
                  tickmode='array', tickvals=_X_VENTANAS, ticktext=_TICKS_VENTANA,
                  tickangle=-45, tickfont=dict(size=8), row=3, col=1)
_fig.update_yaxes(title_text='UITI acumulado',
                  type='log' if GEOMETRIA_014.logs[1] else 'linear',
                  row=3, col=1, secondary_y=False)
# Marcas mas pequenias en los dos ejes que comparten el hueco: cada digito que se
# ahorran es espacio para separar sus rotulos, que estaban a 5 px uno del otro.
_fig.update_yaxes(title_text='Eventos', rangemode='tozero', showgrid=False,
                  title_standoff=2, tickfont=dict(size=9),
                  row=3, col=1, secondary_y=True)

# --- Fila 3, columnas 3-4: el top de variables POR VANO -------------------------------
# Un grupo de barras por vano. Cada traza es una POSICION del ranking (la 1a, la 2a...),
# no una variable: las variables cambian de vano a vano, asi que una traza por variable
# necesitaria tantas como el catalogo entero y casi todas vacias.
# El nombre de la variable va DENTRO de la barra: con cinco grupos de diez barras no hay
# sitio para una leyenda de cincuenta entradas, y el rotulo pegado al dato no obliga a
# cruzarlo. Cual de los tres rotulos posibles -- resumen, inicial o ninguno -- se escribe
# lo decide el repintado segun lo que mida cada barra, y el nombre completo esta siempre en
# la etiqueta del mouse.
# El color codifica la POSICION en el ranking y nada mas. Antes salia de `COLORES_GRUPOS`,
# lo que con diez posiciones dejaba siete del mismo rojo oscuro y, peor, invitaba a leer
# una barra como un grupo de criticidad, que es otra cosa. Una rampa de opacidad sobre UN
# color dice "primera, segunda, tercera" sin pedir prestada la paleta del mapa.
IDX['top_vano'] = [
    _agregar(go.Bar(
        x=[], y=[], name=f'{_p + 1}o', showlegend=False,
        text=[], textposition='inside', insidetextanchor='middle',
        textangle=-90, constraintext='none',
        insidetextfont=dict(size=TAM_FUENTE_BARRA, color='white'),
        marker=dict(color=f'rgba(203,24,29,{0.95 - 0.055 * _p:.2f})',
                    line=dict(width=0.4, color='rgba(60,10,10,0.6)')),
        hovertext=[], hoverinfo='text',
    ), 3, 3) for _p in range(TOP_VARIABLES_POR_VANO)
]
_fig.update_yaxes(title_text='Relevancia variables', title_standoff=2,
                  tickfont=dict(size=9), row=3, col=3)
_fig.update_xaxes(title_text='Vano', tickfont=dict(size=9), row=3, col=3)

# --- Fila 4, columnas 1-2: el grafo de relevancia --------------------------------------
# El peso viaja en un marcador en el PUNTO MEDIO de cada arista y no en el ancho de la
# linea: una sola traza de lineas no puede variar su ancho por segmento, y partirla en una
# traza por arista serian decenas que hay que restilar una por una.
IDX['grafo_aristas'] = _agregar(go.Scattergl(
    x=[], y=[], mode='lines', showlegend=False,
    line=dict(width=1.0, color='rgba(120,110,110,0.45)'), hoverinfo='skip',
), 4, 1)
IDX['grafo_pesos'] = _agregar(go.Scattergl(
    x=[], y=[], mode='markers', showlegend=False,
    marker=dict(size=[], color=[], colorscale='Reds', cmin=0.0, showscale=False,
                line=dict(width=0.4, color='#5b4a48')),
    hovertext=[], hoverinfo='text',
), 4, 1)
IDX['grafo_nodos'] = [
    _agregar(go.Scatter(
        x=[], y=[], mode='markers+text', name=_modalidad, legendgroup='grafo',
        marker=dict(size=7, color=COLORES_MODALIDAD[_modalidad],
                    line=dict(width=0.5, color='#1f2937')),
        text=[], textposition='middle right', textfont=dict(size=7, color='#334155'),
        hovertext=[], hoverinfo='text',
    ), 4, 1) for _modalidad in MODALIDADES_MIL
]
# Sin ejes: una disposicion circular no mide nada en x ni en y. El rango se fija a mano y
# con holgura -- sin ella los rotulos de los nodos del borde salen cortados.
_fig.update_xaxes(visible=False, showticklabels=False, range=[-1.75, 1.75], row=4, col=1)
_fig.update_yaxes(visible=False, showticklabels=False, range=[-1.3, 1.3], row=4, col=1)
# Los ejes del grafo se PREGUNTAN a su traza en vez de escribirse a mano: el numero
# depende de la posicion del subplot en la grilla y del eje secundario de la fila 3, y
# adivinarlo deja el aviso flotando sobre otro panel.
_EJE_X_GRAFO = _fig.data[IDX['grafo_aristas']].xaxis or 'x'
_EJE_Y_GRAFO = _fig.data[IDX['grafo_aristas']].yaxis or 'y'
_fig.add_annotation(text='', xref=f'{_EJE_X_GRAFO} domain', yref=f'{_EJE_Y_GRAFO} domain',
                    x=0.5, y=0.5, showarrow=False, align='center',
                    font=dict(size=11, color='#7a5c58'))
IDX_ANOTACION_GRAFO = len(_fig.layout.annotations) - 1

# --- Fila 4, columnas 3-4: UITI base contra simulado, como violines --------------------
# La MISMA cantidad medida dos veces sobre los MISMOS vanos, lado a lado. Es la lectura
# que cierra el tablero: si la simulacion no mueve la distribucion, no la movio, y eso se
# ve de un vistazo sin tener que comparar dos mapas tramo a tramo.
IDX['violin_base'] = _agregar(go.Violin(
    y=[], name='Base', showlegend=False, fillcolor=COLOR_VIOLIN_BASE, opacity=0.85,
    line=dict(color='#5b4a48', width=1), box_visible=True, meanline_visible=True,
    points='all', jitter=0.25, marker=dict(size=4, opacity=0.6), spanmode='hard',
    hovertemplate='Base -- UITI: %{y:,.2f}<extra></extra>',
), 4, 3)
IDX['violin_simulado'] = _agregar(go.Violin(
    y=[], name='Simulado', showlegend=False, fillcolor=COLOR_VIOLIN_SIMULADO, opacity=0.85,
    line=dict(color='#5b4a48', width=1), box_visible=True, meanline_visible=True,
    points='all', jitter=0.25, marker=dict(size=4, opacity=0.6), spanmode='hard',
    hovertemplate='Simulado -- UITI: %{y:,.2f}<extra></extra>',
), 4, 3)
_fig.update_yaxes(title_text='UITI acumulado (u-hat)', row=4, col=3)

# El aviso del mapa simulado va en coordenadas de PAPEL y no de eje: un subplot de tipo
# `map` no tiene ejes cartesianos a los que anclar una anotacion. El centro sale del
# dominio que `make_subplots` ya calculo, asi que cambiar `row_heights` no lo desalinea.
_dominio_simulado = _fig.layout.map2.domain
_fig.add_annotation(
    text='', xref='paper', yref='paper',
    x=(_dominio_simulado.x[0] + _dominio_simulado.x[1]) / 2.0,
    y=(_dominio_simulado.y[0] + _dominio_simulado.y[1]) / 2.0,
    showarrow=False, align='center', font=dict(size=13, color='#5b4a48'),
    bgcolor='rgba(255,255,255,0.88)', bordercolor='#e4c4c0', borderwidth=1, borderpad=8,
)
IDX_ANOTACION_SIMULADO = len(_fig.layout.annotations) - 1

_fig.update_layout(
    map=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10,
             layers=[CAPA_CAJA_SELECCION]),
    map2=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10,
              layers=CAPAS_CAJA_SIMULADA),
    title=dict(text='Simulador Criticidad'),
    # Margenes explicitos. Los de Plotly por defecto (l=80, r=80, t=100, b=80) se llevaban
    # 160 px de ancho -- medido, el 8,5% de una pantalla de 1.920 -- y a la derecha no hay
    # nada que rotular. El izquierdo NO puede bajar a cero: es donde viven el titulo y las
    # marcas del eje y de los paneles de la primera columna.
    margin=dict(l=52, r=14, t=78, b=44),
    barmode='group', bargap=0.25, bargroupgap=0.05,
    # SIN `width`: con un ancho fijo Plotly ignora el contenedor. Pero dejarlo en None NO
    # basta por si solo -- `autosize` mide el contenedor UNA vez, al montar, y el widget
    # monta antes de que el CSS de la celda lo estire. Lo que cierra el circulo es
    # `responsive` en el config del widget, mas abajo.
    height=1400, autosize=True, template='plotly_white',
    # La leyenda va HORIZONTAL y justo debajo de los mapas. Vertical y a la derecha se
    # llevaba 196 px medidos de ancho para decir siete nombres. `y` sale del dominio del
    # mapa y no de un numero escrito a mano: cambiar `row_heights` mueve los mapas.
    legend=dict(orientation='h', x=0.5, xanchor='center',
                y=_fig.layout.map.domain.y[0] - 0.004, yanchor='top',
                font=dict(size=10), tracegroupgap=22),
)

# El alto en pixeles del panel del top. Va DESPUES de `update_layout` porque el alto de
# la figura se fija alli: leido antes, `_fig.layout.height` todavia es None.
# Es su dominio por el alto de la figura. Es lo que
# permite decidir, barra por barra, si el nombre de la variable cabe escrito adentro --
# el rotulo va girado -90, asi que lo que lo limita es el LARGO de la barra y no su ancho.
# El eje se PREGUNTA a la traza en vez de escribirse a mano: su numero depende del eje
# secundario de la fila 3, y adivinarlo mediria el panel equivocado.
_EJE_Y_TOP = _fig.data[IDX['top_vano'][0]].yaxis or 'y'
_DOMINIO_TOP = _fig.layout[_EJE_Y_TOP.replace('y', 'yaxis', 1)].domain
ALTO_PANEL_TOP_PX = float(_fig.layout.height) * float(_DOMINIO_TOP[1] - _DOMINIO_TOP[0])

# Los indices se verifican al generar: si alguien reordena las trazas, esto falla AQUI y
# no se descubre en silencio al dibujar.
assert len(_fig.data) == 4 + 1 + 1 + 4 + 1 + 4 + 1 + 4 + 2 * MAX_VANOS_ANALISIS \
    + TOP_VARIABLES_POR_VANO + 2 + len(MODALIDADES_MIL) + 2, len(_fig.data)
assert _fig.layout.width is None and _fig.layout.height, (
    'la figura no puede llevar ancho fijo: con uno, Plotly ignora el contenedor. El alto '
    'si es propio. Cuidado: sin ancho fijo NO alcanza -- ver `responsive` abajo.')
# El marcado con color de clase va DESPUES del halo blanco, o el halo lo taparia.
assert min(IDX['marcados_clases']) > IDX['marcados']
assert [_fig.data[i].line.color for i in IDX['marcados_clases']] == COLORES_GRUPOS
# Los equipos son PUNTOS y van despues de todas las lineas: si alguien los adelanta,
# quedan tapados por los tramos.
assert all(_fig.data[i].mode == 'markers'
           for i in (IDX['trafos'], IDX['switches'], IDX['pred_trafos'], IDX['pred_switches']))
assert min(IDX['trafos'], IDX['pred_trafos']) > max(IDX['pred_clases'])
# UN solo negro para la ausencia en los DOS mapas.
assert (_fig.data[IDX['sin_dato']].line.color
        == _fig.data[IDX['pred_sin_dato']].line.color == COLOR_SIN_EVENTO)
assert all(isinstance(_fig.data[i].marker.size, (list, tuple))
           for i in IDX['serie_uiti'] + IDX['serie_eventos']), (
    'marker.size debe ser un array: el punto de la ventana vigente va al triple')
assert all(_fig.data[i].type == 'bar' for i in IDX['top_vano'])
assert _fig.layout.barmode == 'group', 'las barras del top 5 se agrupan POR VANO'
assert all(_fig.data[i].type == 'violin' for i in (IDX['violin_base'], IDX['violin_simulado']))
assert [_fig.data[i].name for i in IDX['grafo_nodos']] == MODALIDADES_MIL
# La caja de seleccion es una CAPA del mapa y no una traza, y va DEBAJO de las trazas: si
# alguien la sube por encima vuelve a comerse el clic que alterna la seleccion. El mapa
# base lleva UNA -- una sola pregunta, "cual elegi" -- y el simulado TRES, una por
# desenlace, porque una capa pinta con un solo color.
assert len(_fig.layout.map.layers) == 1
assert len(_fig.layout.map2.layers) == len(CAMBIOS)
assert all(_capa.below == 'traces'
           for _capa in (*_fig.layout.map.layers, *_fig.layout.map2.layers))
assert ALTO_PANEL_TOP_PX > 0, 'sin alto de panel no se puede decidir si el rotulo cabe'

fig = go.FigureWidget(_fig)
# Un `FigureWidget` NO es fluido por si solo: `width=None` mas `autosize` mas el CSS de la
# celda estiran el DIV, pero plotly sigue DIBUJANDO al ancho que midio al montar -- medido,
# 858 px dentro de un contenedor de 1.935. Su bundle trae un `ResizeObserver` que arregla
# exactamente esto, apagado detras de `config.responsive`. `_config` es un trait
# SINCRONIZADO: lo que se ponga aqui viaja al `newPlot` del navegador y lo enciende.
fig._config = {**(fig._config or {}), 'responsive': True}
assert fig._config.get('responsive') is True, (
    'sin `responsive` el FigureWidget dibuja al ancho de reserva y no al de la celda')
print(f'FigureWidget con {len(fig.data)} trazas en 6 paneles')


In [ ]:
# --- Fila 1: mapa historico con paridad 01.4 + seleccion por casilla o por clic ------
# Tres cosas que el mapa de 01.4 hace y este no hacia: se ENCUADRA sobre el circuito
# elegido (sin eso el circuito queda como un garabato diminuto en un mapa centrado en
# Manizales), dibuja transformadores e interruptores, y da hover por tramo. La cuarta es
# la seleccion: en 01.4 un vano se marca con su casilla O tocandolo en el mapa, y las dos
# vias son EL MISMO estado -- el clic alterna la casilla y deja que todo se rehaga desde
# ahi. Un registro paralelo es como la lista, el mapa y el ranking empiezan a contar
# cosas distintas.


def _seleccion_actual():
    return circuito_widget.value, ventana_widget.value, set(vano_widget.value)


def _plantilla_hover(campo, nombre_clase, ventana, *, marcado=False, extra=''):
    """El tooltip de una traza, como `hovertemplate` y no como texto por punto.

    Lo que varia DENTRO de una traza son solo el fid, el UITI y los eventos, y esos
    viajan crudos en `customdata`. La clase y la ventana son constantes de la traza --
    hay una traza por clase-- asi que van escritas en la plantilla y no se repiten en
    cada punto. Esa diferencia es la que permite densificar: medido sobre el peor
    circuito, repetir la etiqueta formateada cuesta 2,40 MB por capa y esto cuesta 0,66.

    `extra` agrega renglones que SI varian punto a punto y por eso citan `customdata`:
    es como el mapa simulado dice el grupo base de cada vano, que dentro de una traza --
    que es una clase SIMULADA -- cambia de vano a vano.
    """
    return (f'<b>Vano %{{customdata[0]}}</b><br>{ventana["etiqueta"]}: {ventana["periodo"]}'
            f'<br>{campo}: {nombre_clase}{extra}'
            '<br>UITI acumulado: %{customdata[1]}<br>Eventos: %{customdata[2]}'
            + ('<br>(marcado)' if marcado else '') + '<extra></extra>')


def _capas_de_la_seleccion(clases_por_fid, *, campo, nombres_clase,
                           extra_por_fid=None, plantilla_extra=''):
    """Las capas de UN mapa, con el customdata que alimentan el tooltip y el clic.

    `campo` nombra en el tooltip a que pertenece la clase -- "Criticidad original" en
    la fila 1, "Criticidad simulada" en la fila 2.

    `extra_por_fid` agrega columnas al `customdata` de cada punto, para lo que varia
    dentro de una traza y no cabe en la plantilla. `plantilla_extra` es el renglon del
    tooltip que las lee.

    `paso_densificado` interpola vertices cada ~25 m. El hover de una traza de lineas en
    Scattermap se resuelve contra los VERTICES, y los tramos de MVLINSEC traen
    exactamente dos: sin esto, el centro de un vano no muestra etiqueta, y como Plotly
    solo convierte un clic en evento donde hay hover, tampoco se puede marcar tocandolo
    ahi. Es la misma correccion que ya tenia el mapa de 01.
    """
    circuito, ventana_i, marcados = _seleccion_actual()
    geo = GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []})
    ventana = VENTANAS[ventana_i]
    datos = DATOS_VENTANA[ventana_i]

    # Datos CRUDOS por vano; el formato lo pone la plantilla de cada traza.
    datos_por_fid = {fid: datos.get(fid, (0.0, 0)) for fid in geo['fids']}
    if extra_por_fid is not None:
        # La columna extra viaja para TODOS los fids y no solo para los que la tienen:
        # dentro de una traza `customdata` tiene que medir siempre lo mismo, o
        # `%{customdata[3]}` cae en el hueco del vano de al lado.
        datos_por_fid = {fid: (*crudos, *extra_por_fid.get(fid, ('sin dato',)))
                         for fid, crudos in datos_por_fid.items()}
    capas = capas_mapa_historico(
        geo, clases_por_fid, marcados=marcados, datos_por_fid=datos_por_fid,
        marca_extremos=MARCA_VANO, paso_densificado=PASO_VERTICE)
    # Sin celda en la ventana no hay clase, y eso NO es el grupo mas bajo: es la
    # ausencia del dato. Mismo criterio que el tooltip de 01.4.
    sin_dato = 'sin dato'
    capas['plantillas'] = {
        'clases': [_plantilla_hover(campo, nombres_clase[c], ventana,
                                    extra=plantilla_extra) for c in range(4)],
        'sin_dato': _plantilla_hover(campo, sin_dato, ventana, extra=plantilla_extra),
        'marcados_por_clase': [_plantilla_hover(campo, nombres_clase[c], ventana,
                                                marcado=True, extra=plantilla_extra)
                               for c in range(4)],
        'marcados_sin_dato': _plantilla_hover(campo, sin_dato, ventana, marcado=True,
                                              extra=plantilla_extra),
    }
    return capas


def _volcar_capa(traza, capa, plantilla=None):
    """Las tres columnas van juntas SIEMPRE: si `customdata` se desfasa de lat/lon,
    Plotly desalinea el resto de la traza y el clic devuelve el vano equivocado.

    `plantilla` es el `hovertemplate` de la traza. Sin ella la traza no muestra tooltip
    -- es lo que corresponde al halo blanco, que es decoracion y esta debajo de la linea
    de color, que si lo muestra."""
    traza.lat = capa['lat']
    traza.lon = capa['lon']
    traza.customdata = capa['customdata']
    if plantilla is None:
        traza.hoverinfo = 'skip'
    else:
        traza.hovertemplate = plantilla


def _tamanos_ventana_activa(ventana_i):
    """El arreglo de tamanos de marcador de las dos series, con la ventana vigente al
    triple. Es el mismo recurso de la serie del cuaderno 01: `marker.size` es un ARRAY,
    asi que agrandar un punto no obliga a partir la serie en una segunda traza, y mover
    el deslizador solo reescribe once numeros."""
    return (
        [SERIE_TAM_UITI * (FACTOR_PUNTO_ACTIVO if v['i'] == ventana_i else 1)
         for v in VENTANAS],
        [SERIE_TAM_EVENTOS * (FACTOR_PUNTO_ACTIVO if v['i'] == ventana_i else 1)
         for v in VENTANAS],
    )


def _redibujar_mapa_historico(*_ignorado):
    circuito, ventana_i, _marcados = _seleccion_actual()
    capas = _capas_de_la_seleccion(clases_para(circuito, ventana_i),
                                   campo='Criticidad original', nombres_clase=NOMBRES_GRUPOS)
    # El orden de las series sale de la GEOMETRIA y no del orden en que se fueron
    # marcando: asi marcar y desmarcar no baraja los colores bajo la mano.
    marcados_ordenados = [f for f in GEO_POR_CIRCUITO.get(circuito, {}).get('fids', [])
                          if f in _marcados]
    marcados_ordenados = list(dict.fromkeys(marcados_ordenados))[:MAX_VANOS_ANALISIS]
    # La serie describe SOLO los vanos elegidos: sin ninguno marcado queda vacia. Es el
    # mismo criterio que los violines de 01.4 -- una serie sobre el circuito entero y una
    # sobre tres vanos se dibujan igual y no miden lo mismo, asi que caer al circuito
    # cambiaria el sujeto del panel en silencio.
    series = series_temporal_vanos(TABLA, circuito=circuito, fids=marcados_ordenados,
                                   n_ventanas=len(VENTANAS))
    # El grupo de riesgo de cada punto, de UNA sola llamada a la geometria de 01.4 para
    # los hasta 55 puntos dibujados. El repintado corre en cada clic del mapa.
    clases_serie = clases_de_series(series)
    _pl = capas['plantillas']
    with fig.batch_update():
        for _clase in range(4):
            _volcar_capa(fig.data[IDX['clases'][_clase]], capas['clases'][_clase],
                         _pl['clases'][_clase])
            _volcar_capa(fig.data[IDX['marcados_clases'][_clase]],
                         capas['marcados_por_clase'][_clase],
                         _pl['marcados_por_clase'][_clase])
        _volcar_capa(fig.data[IDX['sin_dato']], capas['sin_dato'], _pl['sin_dato'])
        # El halo blanco va SIN tooltip: esta debajo de la linea de color, que ya lo
        # muestra, y dos etiquetas en el mismo punto solo se estorban.
        _volcar_capa(fig.data[IDX['marcados']], capas['marcados'])
        _volcar_capa(fig.data[IDX['marcados_sin_dato']], capas['marcados_sin_dato'],
                     _pl['marcados_sin_dato'])
        # La caja amarilla de lo seleccionado. Sale de la GEOMETRIA y no de las celdas de
        # la ventana: por eso el resaltado sigue puesto al mover el deslizador, incluso
        # sobre un vano que en esa ventana no tiene ni un evento. Se apaga solo al
        # desmarcar el vano -- por su casilla o volviendo a tocarlo en el mapa.
        fig.layout.map.layers[0].source = cajas_seleccion(
            GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []}),
            marcados=_marcados, lado_minimo=LADO_MINIMO_CAJA, margen=MARGEN_CAJA)
        # Fila 3 col 1-2: la serie de tiempo de cada vano elegido, UITI contra el eje
        # izquierdo y eventos contra el derecho. Una ventana sin celda va como `None` y
        # NO como cero: un cero se leeria como "no hubo UITI", y lo que paso es que no
        # hubo medicion. `connectgaps=False` corta la linea ahi.
        _tam_uiti, _tam_eventos = _tamanos_ventana_activa(ventana_i)
        for _cupo in range(MAX_VANOS_ANALISIS):
            _serie = series[_cupo] if _cupo < len(series) else None
            _t_uiti = fig.data[IDX['serie_uiti'][_cupo]]
            _t_eventos = fig.data[IDX['serie_eventos'][_cupo]]
            _sujeto = f'Vano {_serie["fid"]}' if _serie else ''
            _clases = clases_serie[_cupo] if _cupo < len(clases_serie) else []
            # El relleno del punto lleva el grupo de riesgo de ESE vano en ESA ventana;
            # el gris es la ventana sin celda, que no tiene grupo.
            _colores = [COLORES_GRUPOS[c] if c is not None else COLOR_SIN_GRUPO
                        for c in _clases]
            _etiquetas = ([
                f'<b>{_sujeto}</b><br>{VENTANAS[i]["etiqueta"]}: '
                f'{VENTANAS[i]["periodo"]}<br>UITI: {u}<br>Eventos: {e}'
                f'<br>Grupo: {NOMBRES_GRUPOS[c] if c is not None else "sin eventos"}'
                for i, u, e, c in zip(_serie['x'], _serie['uiti'], _serie['eventos'],
                                      _clases)
            ] if _serie else [])
            _t_uiti.x = _serie['x'] if _serie else []
            _t_uiti.y = _serie['uiti'] if _serie else []
            _t_uiti.hovertext = _etiquetas
            _t_uiti.marker.size = _tam_uiti if _serie else []
            _t_uiti.marker.color = _colores
            _t_uiti.name = _sujeto
            _t_eventos.x = _serie['x'] if _serie else []
            _t_eventos.y = _serie['eventos'] if _serie else []
            _t_eventos.hovertext = _etiquetas
            _t_eventos.marker.size = _tam_eventos if _serie else []
            _t_eventos.marker.color = _colores


def _alto_del_mapa_px():
    """El alto en pixeles del subplot de mapa, de su dominio por el alto de la figura.

    Sin esto el zoom salia del span en GRADOS, sin mirar el viewport, y un circuito alto
    quedaba recortado arriba y abajo. El ancho no se pasa: con `autosize` lo decide la
    celda, y encuadrar solo por el alto evita el recorte, que era el defecto."""
    _dom_y = fig.layout.map.domain.y
    return float(fig.layout.height) * float(_dom_y[1] - _dom_y[0])


def _vista_del_circuito(circuito):
    """El encuadre del circuito completo, o None si no tiene geometria. Es la vista de
    referencia de los dos mapas y la que el simulado recupera cuando no hay nada marcado
    sobre lo que acercarse."""
    return centro_y_zoom(GEO_POR_CIRCUITO.get(circuito, {}).get('bounds'),
                         alto_px=_alto_del_mapa_px())


def _aplicar_vista(nombre_mapa, vista):
    if vista is not None:
        getattr(fig.layout, nombre_mapa).center = vista['center']
        getattr(fig.layout, nombre_mapa).zoom = vista['zoom']


def _pintar_circuito(*_ignorado):
    """Lo que depende del CIRCUITO y no de la ventana: equipos y encuadre. Se separa del
    repintado por ventana porque mover la ventana no tiene por que recentrar el mapa --
    en 01.4 el encuadre tambien se hace una sola vez por circuito (`ULTIMO_CENTRADO`)."""
    circuito = circuito_widget.value
    tr = TRAFOS.get(circuito, {'lat': [], 'lon': []})
    sw = SWITCHES.get(circuito, {'lat': [], 'lon': []})
    vista = _vista_del_circuito(circuito)
    with fig.batch_update():
        # Solo la fila 1: los equipos de la fila 2 los pinta el mapa simulado, que antes
        # de la primera simulacion no muestra NADA.
        for _i_tr, _i_sw in ((IDX['trafos'], IDX['switches']),):
            fig.data[_i_tr].lat, fig.data[_i_tr].lon = tr['lat'], tr['lon']
            fig.data[_i_tr].hovertext = ['<b>Transformador</b>'] * len(tr['lat'])
            fig.data[_i_sw].lat, fig.data[_i_sw].lon = sw['lat'], sw['lon']
            fig.data[_i_sw].hovertext = ['<b>Interruptor / switch</b>'] * len(sw['lat'])
        # Los dos arrancan sobre el circuito completo. Despues de simular, el de la
        # derecha se acerca a los vanos marcados (ver `_redibujar_mapa_predicho`); el de
        # la izquierda conserva SIEMPRE esta vista, que es la referencia contra la que se
        # mira el acercamiento.
        for _mapa in ('map', 'map2'):
            _aplicar_vista(_mapa, vista)


_DESC = {'description_width': 'initial'}  # sin esto ipywidgets trunca los rotulos
circuito_widget = widgets.Dropdown(options=CIRCUITOS, description='Circuito',
                                   style=_DESC)
# El rotulo lleva las fechas del intervalo y no solo "V1": una ventana sin sus fechas
# obliga a ir a buscar a que periodo corresponde cada vez que se mueve el deslizador.
ventana_widget = widgets.SelectionSlider(
    options=[(f'{v["etiqueta"]}: {v["periodo"]}', v['i']) for v in VENTANAS],
    description='Ventana', continuous_update=False, style=_DESC,
    layout=widgets.Layout(width='560px'),
)
# Casillas, no SelectMultiple: es la unica forma de que un clic en el mapa alterne el
# MISMO control que el usuario ve, y de que marcar un vano no borre los ya marcados.
# TOPE de MAX_VANOS_ANALISIS: cada vano seleccionado recibe su propia COLUMNA de controles
# mas abajo, y una rejilla de 26 variables por 20 vanos no se lee ni se llena. Al llegar al
# tope las casillas sin marcar se deshabilitan solas, y el clic en el mapa respeta el mismo
# limite -- si no, el mapa seria una puerta trasera para marcar el sexto.
vano_widget = construir_selector_vanos(VANOS_POR_CIRCUITO.get(circuito_widget.value, []),
                                       maximo=MAX_VANOS_ANALISIS)


# Ya no hay "Marcar todos": con el tope en MAX_VANOS_ANALISIS marcaria los primeros cinco
# de la lista, que no es una eleccion que nadie quiera tomar. Queda "Desmarcar", que es el
# camino de vuelta al grano de circuito completo (seleccion vacia).
boton_desmarcar = widgets.Button(description='Desmarcar', button_style='')
boton_desmarcar.on_click(lambda _b: vano_widget.desmarcar_todos())


def _on_circuito_change(_change):
    vano_widget.poblar(VANOS_POR_CIRCUITO.get(circuito_widget.value, []))
    _pintar_circuito()
    _redibujar_mapa_historico()


def _al_hacer_clic(traza, puntos, _estado):
    """Un clic sobre un tramo alterna su vano. El fid sale de `customdata` y no del
    indice del punto: los tramos viajan concatenados con un `None` de separador, asi que
    ese indice cambia con la ventana."""
    fid = fid_de_punto(traza.customdata, getattr(puntos, 'point_inds', ()) or ())
    if fid is not None:
        vano_widget.alternar(fid)


# SOLO el mapa base. La fila 2 es la SALIDA del modelo, no un control: marcar un vano
# desde ahi mezcla "lo que yo elegi" con "lo que el modelo predijo" sobre la misma
# superficie, que es justo la confusion que separa a las dos filas (D2).
# Nota sobre el alcance del clic: plotly solo convierte un clic en evento si en ese punto
# hay hover, y en un `scattermap` de lineas el hover se calcula contra los VERTICES del
# tramo (`scattermap/hover.js`: distancia por punto, radio minimo 3 px, tope
# `layout.hoverdistance`). Antes eso obligaba a tocar el tramo cerca de uno de sus dos
# extremos; ahora `paso_densificado` pone un vertice cada ~25 m, asi que el clic engancha
# en cualquier punto del vano. `hoverdistance` sigue en 30 px, por encima de los 20 por
# defecto, para que el blanco sea generoso sin llegar a marcar un vano lejano.
#
# Se cablean TAMBIEN las capas de vano marcado: quedan dibujadas encima de las de clase,
# asi que son las que recibe el cursor sobre un vano ya marcado. Sin ellas, marcar
# funcionaba y desmarcar tocando el mapa no.
for _i_traza in (IDX['clases'] + IDX['marcados_clases']
                 + [IDX['sin_dato'], IDX['marcados'], IDX['marcados_sin_dato']]):
    fig.data[_i_traza].on_click(_al_hacer_clic)
fig.layout.hoverdistance = 30

# Tier 0 del presupuesto de interactividad (design section A): elegir circuito, mover la
# ventana o marcar un vano no llama al modelo -- sin debounce ni epoch guard, que
# pertenecen al tier 1/2 (fila 2, ranking, boton "Simular"), fuera del alcance de este PR.
circuito_widget.observe(_on_circuito_change, names='value')
ventana_widget.observe(_redibujar_mapa_historico, names='value')
vano_widget.observe(_redibujar_mapa_historico, names='value')

_pintar_circuito()               # equipos y encuadre del circuito inicial
_redibujar_mapa_historico()      # primer dibujo, con la seleccion inicial

In [ ]:
# --- Fila 3, columnas 3-4: el top 5 de variables POR VANO ------------------------------
# El panel mostraba UN ranking, el de la seleccion entera. Con hasta cinco vanos bajo
# estudio eso contesta la pregunta equivocada: dice que variable mueve AL GRUPO, cuando la
# decision de mantenimiento necesita saber cual mueve a ESTE vano, el de la orden de
# trabajo que se esta costeando.
# Sigue sin ser SHAP (decision D5): es un barrido min-max sobre el MISMO modelo y la MISMA
# unidad que el mapa simulado -- la bolsa (vano, ventana) del cuaderno 05 --, y asi se
# llama en todo el cuaderno.
# Cuesta las MISMAS `1 + 2 x knobs_numericos` pasadas que el barrido agregado, no una
# tanda por vano: cada pasada ya devuelve un u-hat por bolsa (ver
# `sensibilidad_minmax_por_vano`). Corre DENTRO del job del boton "Simular" y bajo la misma
# epoca, para que mapa, grafo y top describan siempre la MISMA seleccion.
TOP_VACIO = {}


def _calcular_top_por_vano(seleccion):
    """El barrido, sobre las bolsas que ya resolvio el job de simulacion.

    Recorre `KNOBS_PANEL` y NUNCA `KNOBS` entero: el ranking se queda en los dos
    conjuntos que el panel ofrece -- intervencion y escenario -- por la misma razon por
    la que el panel no los ofrece. Con el catalogo completo entrarian las refutadas, y
    el tablero podria terminar diciendo que la variable mas relevante de un vano es
    `CNT_TRF`, los trafos afectados EN LA FALLA: se mide DESPUES del evento que el
    modelo intenta anticipar. Eso no seria un ranking flojo, seria la flecha del
    analisis al reves, sosteniendo una orden de trabajo que no arregla nada. Tampoco
    entran las de lectura unica: no se puede rankear por relevancia lo que no se deja
    mover.
    """
    return sensibilidad_minmax_por_vano(
        MIL, X_INST, seleccion=seleccion, feature_names=FEATURES_MIL, knobs=KNOBS_PANEL,
        top=TOP_VARIABLES_POR_VANO, label_encoders=label_encoders,
        max_values_imputed=max_values_imputed,
    )


def _pintar_top_por_vano(por_vano):
    """Repaint puro, cero pasadas del modelo.

    Un grupo de barras por vano. Cada TRAZA es una posicion del ranking -- la 1a, la 2a...
    -- y no una variable: las variables cambian de vano a vano, asi que una traza por
    variable necesitaria tantas como el catalogo entero y casi todas vacias.
    El nombre va DENTRO de la barra: con cinco grupos de diez barras no hay sitio para una
    leyenda de cincuenta entradas, y el rotulo pegado al dato no obliga a cruzarlo.

    Cual de los tres rotulos se escribe lo decide `rotulo_en_barra` con el largo de CADA
    barra en pixeles: el resumen si cabe, sus iniciales si no, y nada antes que un texto
    cortado que se monte sobre la barra vecina. El nombre completo esta siempre en la
    etiqueta del mouse, que es donde se resuelve la duda.
    """
    vanos = list(por_vano)
    # El rango del eje se decide ANTES de escribir las barras: el rotulo de cada una
    # depende de cuantos pixeles mide, y eso solo se sabe con el rango ya fijado. Con las
    # trazas vacias un eje lineal autoescala a [-1, 4] y muestra marcas negativas para una
    # magnitud que no puede serlo.
    _tope = max((f['magnitud'] for filas in por_vano.values() for f in filas), default=0.0)
    _rango = _tope * 1.15 if _tope > 0 else 1.0
    _px_por_unidad = ALTO_PANEL_TOP_PX / _rango
    with fig.batch_update():
        fig.update_yaxes(range=[0, _rango], row=3, col=3)
        for _posicion in range(TOP_VARIABLES_POR_VANO):
            _traza = fig.data[IDX['top_vano'][_posicion]]
            _x, _y, _texto, _hover = [], [], [], []
            for _fid in vanos:
                _filas = por_vano.get(_fid, [])
                if _posicion >= len(_filas):
                    continue
                _fila = _filas[_posicion]
                _x.append(_fid)
                _y.append(_fila['magnitud'])
                _texto.append(rotulo_en_barra(_fila['label'],
                                              _fila['magnitud'] * _px_por_unidad))
                _hover.append(
                    f'<b>Vano {_fid}</b><br>{_posicion + 1}o: {_fila["label"]}'
                    f'<br>Sensibilidad min-max: {_fila["magnitud"]:.4g}'
                    f'<br>Al maximo: {_fila["direccion_maximo"]}'
                    f'<br>Al minimo: {_fila["direccion_minimo"]}'
                )
            _traza.x, _traza.y = _x, _y
            _traza.text, _traza.hovertext = _texto, _hover


def _pintar_violines(tabla_simulada):
    """Fila 4, columnas 3-4: el UITI de los MISMOS vanos, antes y despues de simular.

    Es la lectura que cierra el tablero: si la simulacion no mueve la distribucion, no la
    movio, y eso se ve de un vistazo sin comparar dos mapas tramo a tramo. Con pocos vanos
    un violin es casi una linea, por eso van tambien los puntos (`points='all'`): con tres
    o cuatro datos lo honesto es mostrarlos, no dibujar una densidad que no existe.
    """
    with fig.batch_update():
        if tabla_simulada is None or tabla_simulada.empty:
            fig.data[IDX['violin_base']].y = []
            fig.data[IDX['violin_simulado']].y = []
        else:
            fig.data[IDX['violin_base']].y = list(tabla_simulada['u_base'])
            fig.data[IDX['violin_simulado']].y = list(tabla_simulada['u_simulado'])


_pintar_top_por_vano(TOP_VACIO)   # vacio hasta el primer "Simular"
_pintar_violines(None)


In [ ]:
# --- Fila 2: mapa "Criticidad Simulada" + boton "Simular" (design section A, decision D2)
# El boton es el UNICO disparador y hace TRES cosas de una sola vez, bajo la misma epoca:
# el mapa simulado, el grafo reconstruido de la seleccion y el barrido de importancia de
# la celda anterior. Ya no hay alternador base/simulado/delta: el mapa de la fila 2
# muestra SIEMPRE la clase simulada, que es lo que el boton promete.
#
# El mapa y el grafo salen del modelo MIL del cuaderno 05, que puntua BOLSAS: una bolsa es
# una celda (vano, ventana) y su clase sale de `asignar_clase(n_obs OBSERVADO, u-hat
# predicho)` sobre la geometria KMeans de 01.4 -- la MISMA con la que se pinta el mapa
# base, que es por lo que los dos mapas comparten paleta por construccion y no por
# convencion. `n_obs` nunca se simula: es un eje del espacio que define la clase.
# Debounce asincronico (design section A): `asyncio.ensure_future` + cancelacion en el
# propio event loop del kernel, NUNCA `threading.Timer` -- ipykernel enruta la salida de
# los widgets con el parent header thread-local, asi que una escritura desde un hilo en
# segundo plano cae en la celda equivocada. `_EPOCA` es el guard de epoca: cualquier evento
# que invalide un job en vuelo la avanza y la escritura tardia se descarta.
from chec_local_interpreter.vano_app_015 import (
    DEBOUNCE_SEGUNDOS,
    ESTADO_BASE,
    ESTADO_SIMULADO,
    aplicar_si_vigente,
    clases_por_fid_para_estado,
    siguiente_epoca,
)
from chec_local_interpreter.vano_widgets import widget_for_knob

# El estado vacio se PIDE a la funcion en vez de escribirlo a mano: escrito a mano se
# desincroniza en cuanto `trazas_grafo` agrega una columna, que es exactamente lo que
# paso al sumarle el indice de modalidad a cada nodo.
GRAFO_VACIO = trazas_grafo(np.zeros((1, 1)), [''])

_EPOCA = 0
_tarea_pendiente_simular = None
_ultimo_resultado_simulacion = None   # DataFrame de simulate_explicit_overrides, o None
_ultima_seleccion_simulada = None     # (circuito, ventana_i) al que corresponde ese resultado

STATUS = widgets.HTML(
    'Sin simular todavia -- elige variables (opcional) y presiona "Simular".'
)

# El panel NO ofrece las variables refutadas. Mientras estuvieran en la lista, el
# tablero las presentaba como equivalentes a la poda o a la puesta a tierra, y tarde o
# temprano alguien mueve las coordenadas de un vano creyendo que eso es un escenario.
# Quitarlas del panel no las saca de la SIMULACION: un override solo se escribe si se
# fija, asi que entran al modelo con el valor OBSERVADO de cada vano, que es lo que
# corresponde. Lo unico que se pierde es poder moverlas.
KNOBS_PANEL = knobs_simulables(KNOBS)
KNOBS_BLOQUEADOS = knobs_bloqueados(KNOBS)
_knobs_por_id = {k.id: k for k in KNOBS_PANEL}
# Casillas y no `SelectMultiple`, por el mismo motivo que la lista de vanos: en un
# `SelectMultiple` un clic sin ctrl borra todo lo ya elegido, y aqui justamente se quiere
# simular VARIAS variables a la vez. Cada casilla es independiente y `value` sigue siendo
# la tupla de knob ids, asi que `_reconstruir_controles_knob` no se entera del cambio.
# CUATRO columnas: dos para lo que se puede hacer y dos para lo que se quiere
# anticipar. Una lista corrida de dieciocho casillas obliga a recordar el veredicto de
# cada variable para saber a cual de las dos preguntas pertenece -- "que obra hago" y
# "que pasa si" --; en columnas eso lo dice la posicion.
COLUMNAS_KNOBS = columnas_panel(KNOBS_PANEL)
knob_selector_widget = construir_selector_casillas(
    columnas=[(titulo, [(k.label, k.id) for k in knobs]) for titulo, knobs in COLUMNAS_KNOBS],
    titulo='', alto='210px', ancho_casilla='215px',
    layout=widgets.Layout(width='100%'),
)
# Se NOMBRAN en vez de dejarlas desaparecer: una lista que se acorta sin explicacion se
# lee como que faltan variables, no como una decision.
AVISO_BLOQUEADOS = widgets.HTML(
    '' if not KNOBS_BLOQUEADOS else
    '<span style="font-size:12px;color:#5b4a48;">No simulables: <b>'
    + ', '.join(k.label for k in KNOBS_BLOQUEADOS) + '</b>. '
    'Entran a la simulacion con el valor observado de cada vano, pero no se pueden '
    'mover: la tabla de arriba dice por que. </span>')
controles_knob_box = widgets.VBox([])
# {fid o GRANO_CIRCUITO: {knob_id: widget}}. Una COLUMNA por vano, con el vano escrito
# encima: sin ese encabezado cinco columnas de deslizadores identicos son indistinguibles.
_controles_por_vano = {}
GRANO_CIRCUITO = '(todo el circuito)'


def _seleccion_de_bolsas():
    """Las bolsas de la seleccion activa, o None si no hay ninguna. Es lo que hace falta
    para saber que vanos tienen columna y en que valor arranca cada control."""
    circuito, ventana_i, marcados = _seleccion_actual()
    seleccion = seleccionar_bolsas(BAG_INDEX, circuito=circuito,
                                   ventana=VENTANAS[ventana_i]['etiqueta'],
                                   marcados=marcados)
    return seleccion if seleccion['n_bolsas'] else None


def _valores_iniciales(seleccion, marcados):
    """En que valor abre cada control. La regla: el valor ACTUAL de esa variable para ESE
    vano en la ventana activa -- mediana si el vano trae varias instancias, moda si la
    variable es categorica (ver `valores_actuales_por_vano`).

    Un control que abriera en un valor por defecto pediria volver a teclear un dato que el
    modelo ya tiene, y peor: cualquier variable que se dejara quieta simularia al vano en
    un valor que nunca fue el suyo.

    Sin vanos marcados el grano es el circuito completo, y entonces hay UNA columna: todas
    las instancias se resumen juntas, que es exactamente lo que el override global escribe.
    """
    if seleccion is None:
        return {}
    X_sel = X_INST[seleccion['filas']]
    if marcados:
        return valores_actuales_por_vano(
            X_sel, FEATURES_MIL, instance_bag=seleccion['instance_bag'],
            fids=seleccion['fid'], knobs=KNOBS_PANEL, label_encoders=label_encoders,
        )
    return valores_actuales_por_vano(
        X_sel, FEATURES_MIL,
        instance_bag=np.zeros(len(X_sel), dtype=np.int64), fids=[GRANO_CIRCUITO],
        knobs=KNOBS_PANEL, label_encoders=label_encoders,
    )


def _control_con_valor(knob, valor):
    """El control del knob, abierto en `valor`. Un valor fuera de los limites del
    deslizador no se fuerza: se recorta, porque `FloatSlider` lanza si el valor cae fuera
    de [min, max] y tumbar el panel entero por un decimal no vale la pena."""
    control = widget_for_knob(knob)
    if valor is None:
        return control
    if knob.kind == 'numeric':
        lo, hi = knob.bounds
        control.value = float(min(max(float(valor), lo), hi))
    elif knob.kind == 'categorical' and valor in (knob.categories or ()):
        control.value = valor
    return control


def _reconstruir_controles_knob(_change=None):
    """La rejilla: filas = variables elegidas, columnas = vanos elegidos.

    Se rehace cuando cambia CUALQUIERA de las dos listas, y tambien al mover circuito o
    ventana: el valor inicial de cada control es el del vano EN ESA VENTANA, asi que una
    rejilla que sobreviviera al deslizador estaria mostrando los valores de otra.
    """
    global _controles_por_vano
    _controles_por_vano = {}
    knob_ids = list(knob_selector_widget.value)
    _circuito, _ventana_i, marcados = _seleccion_actual()
    seleccion = _seleccion_de_bolsas()
    valores = _valores_iniciales(seleccion, marcados)

    if not knob_ids:
        controles_knob_box.children = [widgets.HTML(
            '<span style="font-size:12px;color:#5b4a48;">Elige arriba las variables a '
            'modificar. Cada vano marcado recibe su propia columna de controles.</span>')]
        return
    if seleccion is None:
        controles_knob_box.children = [widgets.HTML(
            '<span style="font-size:12px;color:#5b4a48;">Esta seleccion no tiene celdas '
            '(vano x ventana) en la ventana activa: no hay valores desde donde '
            'arrancar.</span>')]
        return

    # El ORDEN de las columnas sale de la geometria y no del orden en que se fueron
    # marcando: asi marcar y desmarcar no baraja las columnas bajo la mano.
    columnas_fid = ([f for f in seleccion['fid'] if f in valores] if marcados
                    else [GRANO_CIRCUITO])
    columnas = []
    for fid in columnas_fid:
        controles = {}
        for knob_id in knob_ids:
            knob = _knobs_por_id[knob_id]
            controles[knob_id] = _control_con_valor(knob, valores.get(fid, {}).get(knob_id))
        _controles_por_vano[fid] = controles
        encabezado = widgets.HTML(
            f'<div style="font-weight:600;border-bottom:2px solid rgb(203,24,29);'
            f'padding-bottom:2px;margin-bottom:4px;">{fid}</div>')
        columnas.append(widgets.VBox(
            [encabezado, *controles.values()],
            layout=widgets.Layout(margin='0 14px 0 0', align_items='flex-start')))

    # Un vano marcado SIN celda en la ventana activa no tiene de donde sacar un valor
    # inicial, asi que no recibe columna. Se dice: medido, con 5 vanos marcados la rejilla
    # armaba 4 columnas y el quinto desaparecia sin que nada en pantalla lo explicara.
    sin_columna = [f for f in marcados if f not in _controles_por_vano]
    aviso_faltantes = ('' if not sin_columna else
                       f'<br>Sin columna: {", ".join(sorted(sin_columna))} -- '
                       f'{"ese vano no tiene" if len(sin_columna) == 1 else "esos vanos no tienen"} '
                       'eventos en la ventana activa, asi que no hay valor actual desde '
                       'donde arrancar. La simulacion tampoco los puntua.')
    pie = widgets.HTML(
        '<span style="font-size:12px;color:#5b4a48;">Cada control abre en el valor actual '
        'de esa variable para ese vano en la ventana activa (mediana de sus instancias; '
        f'moda si es categorica).{aviso_faltantes}</span>')
    # `flex_flow='row wrap'`: con cinco columnas y una variable de nombre largo la fila se
    # pasa del ancho del panel, y sin el wrap las ultimas columnas quedaban cortadas.
    controles_knob_box.children = [
        widgets.Box(columnas, layout=widgets.Layout(display='flex', flex_flow='row wrap',
                                                    align_items='flex-start',
                                                    width='100%')),
        pie,
    ]


knob_selector_widget.observe(_reconstruir_controles_knob, names='value')
# La rejilla depende de la ventana y del circuito por sus VALORES INICIALES, no solo por
# que vanos existen: mover el deslizador tiene que reabrir los controles en los valores de
# la ventana nueva.
vano_widget.observe(_reconstruir_controles_knob, names='value')
ventana_widget.observe(_reconstruir_controles_knob, names='value')
circuito_widget.observe(_reconstruir_controles_knob, names='value')

boton_simular = widgets.Button(description='Simular', button_style='primary')


_CAPA_VACIA = {'lat': [], 'lon': [], 'hovertext': [], 'customdata': []}


def _redibujar_mapa_predicho(*_ignorado):
    """Repaint puro, CERO llamadas al modelo.

    Antes de la primera simulacion de la seleccion activa el mapa no se dibuja: ni
    tramos, ni equipos, ni leyenda -- solo el aviso de que hay que presionar "Simular".
    Un mapa completo pintado de "aun no simulado" ocupa el mismo lugar y tiene la misma
    forma que un resultado, y esa es justamente la confusion que la fila 2 no puede
    permitirse (D2).

    Con resultado, cada vano va del color del grupo que el simulador le predijo -- la
    MISMA paleta del mapa base, porque es la misma geometria -- y NEGRO todo lo demas
    (vano sin evento en la ventana, o no seleccionado), igual que la estructura del
    circuito en 01.4. Ademas de eso, este mapa hace dos cosas que el base no:

    - encierra a cada vano simulado en un recuadro cuyo COLOR es el desenlace -- bajo,
      se quedo igual o subio de grupo --, en tres capas de `layout.map2.layers`;
    - se ACERCA a los vanos marcados, en vez de quedarse en el encuadre del circuito.
    """
    circuito, ventana_i, _marcados = _seleccion_actual()
    geo = GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []})
    hay_resultado = (
        _ultimo_resultado_simulacion is not None
        and _ultima_seleccion_simulada == (circuito, ventana_i)
    )
    if not hay_resultado:
        with fig.batch_update():
            for _i in (IDX['pred_clases'] + [IDX['pred_sin_dato'],
                                             IDX['pred_trafos'], IDX['pred_switches']]):
                fig.data[_i].lat, fig.data[_i].lon = [], []
                fig.data[_i].hovertext = []
                fig.data[_i].showlegend = False
            # Las tres cajas se apagan juntas: un recuadro de desenlace sobre un mapa que
            # no muestra ningun resultado afirmaria un cambio que nadie calculo.
            for _capa in fig.layout.map2.layers:
                _capa.source = {'type': 'FeatureCollection', 'features': []}
            _aplicar_vista('map2', _vista_del_circuito(circuito))
            fig.layout.annotations[IDX_ANOTACION_SIMULADO].text = (
                'El mapa simulado aparece al presionar <b>Simular</b>.'
            )
        return

    clases_por_fid = clases_por_fid_para_estado(_ultimo_resultado_simulacion, ESTADO_SIMULADO)
    # El grupo BASE viaja como una columna mas del `customdata` y no en la plantilla:
    # dentro de una traza -- que es UNA clase simulada -- el grupo base cambia de vano a
    # vano. Sin los dos numeros en la misma etiqueta, saber si el vano mejoro obliga a
    # cruzar al mapa de al lado y acordarse del color.
    clases_base = clases_por_fid_para_estado(_ultimo_resultado_simulacion, ESTADO_BASE)
    capas = _capas_de_la_seleccion(
        clases_por_fid, campo='Criticidad simulada', nombres_clase=NOMBRES_GRUPOS,
        extra_por_fid={_fid: (NOMBRES_GRUPOS[_c],) for _fid, _c in clases_base.items()},
        plantilla_extra='<br>Criticidad base: %{customdata[3]}')
    _pl = capas['plantillas']
    tr = TRAFOS.get(circuito, {'lat': [], 'lon': []})
    sw = SWITCHES.get(circuito, {'lat': [], 'lon': []})
    with fig.batch_update():
        for _clase in range(4):
            _volcar_capa(fig.data[IDX['pred_clases'][_clase]], capas['clases'][_clase],
                         _pl['clases'][_clase])
            # NO vuelve a la leyenda: el mapa simulado usa la MISMA geometria KMeans
            # que el base, asi que sus cuatro clases ya estan nombradas alli. Repetirlas
            # agregaba un renglon a la leyenda horizontal que caia sobre los titulos de
            # la fila 3 -- medido, 22 px de solape.
            fig.data[IDX['pred_clases'][_clase]].showlegend = False
        # Negro: sin evento en la ventana, o fuera de la seleccion simulada. Un vano que
        # el simulador no puntuo no tiene clase, y la ausencia no es la clase mas baja.
        # Este mapa NO lleva halo de marcado: lo coloreado ES la seleccion, asi que un
        # halo encima no distinguiria nada que el color no diga ya.
        _volcar_capa(fig.data[IDX['pred_sin_dato']], capas['sin_dato'], _pl['sin_dato'])
        fig.data[IDX['pred_sin_dato']].name = 'Sin evento / no simulado'
        fig.data[IDX['pred_sin_dato']].line.color = COLOR_SIN_EVENTO
        fig.data[IDX['pred_sin_dato']].showlegend = False
        fig.data[IDX['pred_trafos']].lat, fig.data[IDX['pred_trafos']].lon = tr['lat'], tr['lon']
        fig.data[IDX['pred_trafos']].hovertext = ['<b>Transformador</b>'] * len(tr['lat'])
        fig.data[IDX['pred_switches']].lat, fig.data[IDX['pred_switches']].lon = sw['lat'], sw['lon']
        fig.data[IDX['pred_switches']].hovertext = ['<b>Interruptor / switch</b>'] * len(sw['lat'])
        # El recuadro de este mapa dice QUE LE PASO al vano y no cual elegi -- eso ya lo
        # dice el de la izquierda, sobre el mismo vano. Verde si bajo de grupo, amarillo
        # si se quedo igual, rojo si subio; el amarillo es el mismo del mapa base porque
        # "no cambio" es justo el estado en que los dos mapas dicen lo mismo. Un vano
        # marcado sin celda en la ventana no recibe caja: la simulacion no lo puntuo, asi
        # que no tiene desenlace que pintar.
        # Marcados que la simulacion PUNTUO. No es lo mismo que los marcados: marcar un
        # vano mas despues de simular no lo mete en el resultado, y ni la caja ni el
        # encuadre pueden seguirlo hasta que se vuelva a presionar "Simular".
        _simulados = set(_ultimo_resultado_simulacion['FID_VANO'].astype(str))
        _marcados_simulados = [f for f in _marcados if f in _simulados]
        _cajas = cajas_por_cambio_de_grupo(
            geo, _ultimo_resultado_simulacion, marcados=_marcados_simulados,
            lado_minimo=LADO_MINIMO_CAJA, margen=MARGEN_CAJA)
        for _cambio, _i_capa in IDX_CAPA_CAMBIO.items():
            fig.layout.map2.layers[_i_capa].source = _cajas[_cambio]
        # Y se ACERCA a los vanos marcados. Los dos mapas dejan de compartir vista a
        # proposito: una vez simulado, la pregunta es que le paso a ESOS vanos, y buscarlos
        # otra vez dentro del circuito entero es trabajo que el tablero puede ahorrar. El
        # de la izquierda conserva el encuadre del circuito, que queda como la referencia.
        # Sin vanos marcados -- grano de circuito completo -- no hay sobre que acercarse y
        # se vuelve a ese mismo encuadre, en vez de quedarse en el de la seleccion anterior.
        _aplicar_vista('map2', centro_y_zoom(bounds_de_fids(geo, _marcados_simulados),
                                             alto_px=_alto_del_mapa_px())
                       or _vista_del_circuito(circuito))
        fig.layout.annotations[IDX_ANOTACION_SIMULADO].text = ''


def _limpiar_resultado_simulacion(_change=None):
    """Circuito o ventana cambiaron: el ultimo resultado ya NO corresponde a la
    seleccion activa -- se descarta (fila 2 vuelve a "Aun no simulado" y el panel de
    importancia se vacia) en vez de mostrar la corrida de OTRA seleccion, que violaria
    la regla anti-confusion (D2)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada, _EPOCA
    _ultimo_resultado_simulacion = None
    _ultima_seleccion_simulada = None
    _EPOCA = siguiente_epoca(_EPOCA)  # invalida cualquier job en vuelo
    _redibujar_mapa_predicho()
    _pintar_grafo(None)
    _pintar_top_por_vano(TOP_VACIO)
    _pintar_violines(None)


def _pintar_grafo(grafo):
    """Repaint puro del panel del grafo. Un grafo ANULADO no se dibuja a medias: se
    vacian las trazas y se dice por que. `estadistico_colapso` anula cuando las
    compuertas no varian entre vanos -- y su veredicto incluye `effective_rank <= 1`,
    que con menos de 3 vanos se cumple por construccion (la matriz centrada de 1 o 2
    filas tiene rango 1). Dibujar igual seria presentar el grafo experto FIJO como si
    lo hubiera estimado esta seleccion."""
    if grafo is None:
        trazas, mensaje = GRAFO_VACIO, 'Presiona "Simular" para estimar el grafo.'
    elif grafo['voided']:
        trazas = GRAFO_VACIO
        mensaje = (f'Grafo no estimable: las compuertas no varian entre los '
                   f'{grafo["n_vanos"]} vanos de la seleccion.<br>'
                   '<sup>Hacen falta al menos 3 vanos con comportamiento distinto.</sup>')
    else:
        trazas, mensaje = trazas_grafo(grafo['matriz'], FEATURES_MIL), ''

    with fig.batch_update():
        fig.data[IDX['grafo_aristas']].x = trazas['aristas']['x']
        fig.data[IDX['grafo_aristas']].y = trazas['aristas']['y']
        _pesos = trazas['pesos']
        fig.data[IDX['grafo_pesos']].x = _pesos['x']
        fig.data[IDX['grafo_pesos']].y = _pesos['y']
        fig.data[IDX['grafo_pesos']].hovertext = _pesos['hovertext']
        # El tamano codifica el peso relativo DE ESTA seleccion: los pesos absolutos
        # cambian dos ordenes de magnitud entre ventanas y un tamano fijo por valor
        # dejaria el panel vacio o saturado segun cual se mire.
        _maximo = max(_pesos['peso'], default=0.0) or 1.0
        fig.data[IDX['grafo_pesos']].marker.size = [4 + 10 * (p / _maximo) for p in _pesos['peso']]
        fig.data[IDX['grafo_pesos']].marker.color = list(_pesos['peso'])
        # Un nodo por variable, con su NOMBRE al lado y el color de su modo. El rotulo
        # se manda hacia afuera del circulo (a la derecha en la mitad derecha, a la
        # izquierda en la izquierda) para que no se monte sobre las aristas.
        _nodos = trazas['nodos']
        for _i_traza, _modalidad in zip(IDX['grafo_nodos'], MODALIDADES_MIL):
            _cuales = [k for k, col in enumerate(_nodos['indice'])
                       if col in COLUMNAS_MODALIDAD[_modalidad]]
            _traza_nodo = fig.data[_i_traza]
            _traza_nodo.x = [_nodos['x'][k] for k in _cuales]
            _traza_nodo.y = [_nodos['y'][k] for k in _cuales]
            _traza_nodo.text = [_nodos['texto'][k] for k in _cuales]
            _traza_nodo.textposition = ['middle right' if _nodos['x'][k] >= 0
                                        else 'middle left' for k in _cuales]
            _traza_nodo.hovertext = [f'<b>{_nodos["texto"][k]}</b><br>Modo: {_modalidad}'
                                     for k in _cuales]
        fig.layout.annotations[IDX_ANOTACION_GRAFO].text = mensaje


def _simular(epoca_job):
    """Computo pesado -- bloqueante dentro de la corutina (design section A: un job ya
    iniciado no se puede interrumpir). Mapa simulado, grafo e importancia, en ese orden y
    en el mismo job. Guarda y repinta SOLO si `epoca_job` sigue vigente al terminar
    (epoch guard)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
    circuito, ventana_i, marcados = _seleccion_actual()
    seleccion = seleccionar_bolsas(BAG_INDEX, circuito=circuito,
                                   ventana=VENTANAS[ventana_i]['etiqueta'],
                                   marcados=marcados)
    if seleccion['n_bolsas'] == 0:
        aplicar_si_vigente(
            lambda: setattr(STATUS, 'value',
                            'Sin bolsas (vano x ventana) para esta seleccion.'),
            epoca_job=epoca_job, epoca_actual=lambda: _EPOCA,
        )
        return

    # Cada columna de la rejilla es un vano, y cada vano lleva SUS valores a sus propias
    # instancias. Con la escritura global anterior el ultimo vano pisaba a todos los demas
    # y la simulacion contestaba por un escenario que nadie habia pedido.
    # Sin vanos marcados el grano es el circuito completo y hay una sola columna: ahi el
    # override vuelve a ser global, que es lo que corresponde a una pregunta sobre todo el
    # circuito.
    # `KNOBS` completo y no `KNOBS_PANEL`: el diccionario solo se usa para resolver
    # que features toca cada knob, y solo llegan aqui los que el panel ofrecio.
    por_vano = {
        fid: expand_knob_overrides(
            {knob_id: control.value for knob_id, control in controles.items()}, KNOBS)
        for fid, controles in _controles_por_vano.items()
    }
    global_ = por_vano.pop(GRANO_CIRCUITO, None)

    t0 = time.perf_counter()
    resultado, metadata = simular_bolsas(
        MIL, X_INST, seleccion=seleccion, feature_names=FEATURES_MIL,
        overrides=global_ if global_ is not None else None,
        overrides_por_vano=por_vano or None,
        label_encoders=label_encoders, max_values_imputed=max_values_imputed,
    )
    # El grafo se estima sobre las features OBSERVADAS de la seleccion, no sobre las
    # simuladas: describe a estos vanos, no al escenario hipotetico.
    gates = gates_de_bolsas(MIL, X_INST[seleccion['filas']], seleccion['instance_bag'],
                            seleccion['n_bolsas'])
    grafo = grafo_de_gates(gates, MIL.model.edge_index, n_features=len(FEATURES_MIL))
    # El top por vano reusa la MISMA seleccion de bolsas que acaba de puntuar el mapa: no
    # se vuelve a resolver, que era lo que hacia el cache por (circuito, ventana, marcados).
    top_por_vano = _calcular_top_por_vano(seleccion)
    duracion = time.perf_counter() - t0

    def _escribir():
        global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
        _ultimo_resultado_simulacion = resultado
        _ultima_seleccion_simulada = (circuito, ventana_i)
        grano = f'{len(marcados)} vanos marcados' if marcados else 'todo el circuito'
        cambian = int((resultado['delta_riesgo_ordinal'] != 0).sum())
        avisos = f' | {len(metadata["avisos"])} avisos' if metadata['avisos'] else ''
        STATUS.value = (
            f'{duracion:.2f} s | MIL sobre {metadata["n_vanos"]} bolsas '
            f'({metadata["n_instancias"]:,} eventos) | '
            f'{len(metadata["variables_aplicadas"])} variables aplicadas | '
            f'{cambian} vanos cambian de clase | grafo sobre {grano}'
            f' | top {TOP_VARIABLES_POR_VANO} de {len(top_por_vano)} vanos{avisos}'
        )
        _redibujar_mapa_predicho()
        _pintar_grafo(grafo)
        _pintar_top_por_vano(top_por_vano)
        _pintar_violines(resultado)

    aplicar_si_vigente(_escribir, epoca_job=epoca_job, epoca_actual=lambda: _EPOCA)


def _programar_simulacion(*_ignorado):
    global _EPOCA, _tarea_pendiente_simular
    if _tarea_pendiente_simular is not None and not _tarea_pendiente_simular.done():
        _tarea_pendiente_simular.cancel()
    _EPOCA = siguiente_epoca(_EPOCA)
    epoca_job = _EPOCA
    STATUS.value = 'Simulando...'

    async def _tarea():
        try:
            await asyncio.sleep(DEBOUNCE_SEGUNDOS)
        except asyncio.CancelledError:
            return
        _simular(epoca_job)

    _tarea_pendiente_simular = asyncio.ensure_future(_tarea())


boton_simular.on_click(_programar_simulacion)
circuito_widget.observe(_limpiar_resultado_simulacion, names='value')
ventana_widget.observe(_limpiar_resultado_simulacion, names='value')
vano_widget.observe(_redibujar_mapa_predicho, names='value')  # solo redibuja el halo marcado

_redibujar_mapa_predicho()  # primer dibujo: sin simulacion todavia -> "Aun no simulado"
_pintar_grafo(None)
_reconstruir_controles_knob()   # la rejilla arranca con su aviso, no vacia


In [ ]:
# --- El panel, ARRIBA y del ancho de la figura (paridad 01.4) -----------------------
# Una sola columna, en el orden en que se usa: circuito -> ventana -> vanos -> variables
# del simulador -> el control de cada variable elegida -> "Simular" -> estado. Cada paso
# depende del anterior, asi que apilarlos evita el zigzag de un flex-wrap donde el boton
# podia quedar antes de los deslizadores que lo alimentan.
#
# El estilo va por CSS y no por `Layout` porque ipywidgets 8 no expone `background`,
# `box-sizing` ni `gap` como traits -- solo `border`, `padding`, `margin` y el flexbox
# basico. `add_class` es la via soportada para lo demas.
ESTILO = widgets.HTML('''
<style>
  .panel-v15 {
    box-sizing: border-box;
    border-radius: 6px; background: #fdf7f6; color: #2b2b2b; font-size: 13px;
  }
  /* Cada grupo, un renglon completo: el panel es una columna, no una grilla. */
  .panel-v15 .grupo-v15 { margin: 0 0 10px 0; width: 100%; }
  .panel-v15 .titulo-v15 { font-weight: 600; margin-bottom: 2px; }
  /* La figura no lleva ancho fijo: sin esto se quedaria en el ancho intrinseco que
     plotly.js calcula al montar, en vez de ocupar la celda. */
  .app-v15, .app-v15 > .widget-vbox, .app-v15 .js-plotly-plot,
  .app-v15 .plot-container, .app-v15 .svg-container { width: 100% !important; }
  /* La lista compacta de 01.4: letra 12px, muchas casillas por renglon y scroll propio en
     vez de estirar el panel cuando el circuito tiene cientos de vanos.
     El ancho de cada casilla NO se toca aqui: viaja como estilo inline desde su `Layout`
     y le ganaria a esta hoja igual. */
  .lista-vanos, .lista-variables { font-size: 12px; }
  .lista-vanos .widget-checkbox label,
  .lista-variables .widget-checkbox label { white-space: nowrap; font-weight: 400; }
</style>''')


def _grupo(*hijos):
    """Un bloque del panel: su rotulo y sus controles juntos, como los `div` de 01.4."""
    caja = widgets.VBox(list(hijos), layout=widgets.Layout(align_items='flex-start',
                                                           width='100%'))
    caja.add_class('grupo-v15')
    return caja


def _titulo(texto):
    return widgets.HTML(f'<span class="titulo-v15">{texto}</span>')


vano_widget.caja.add_class('lista-vanos')
knob_selector_widget.caja.add_class('lista-variables')

# El panel se alinea con el AREA DE DIBUJO de la figura, no con su borde. Los dos
# ocupan el mismo ancho, pero la figura reserva margen para los rotulos de los ejes de
# la primera columna, asi que sus paneles empiezan mas adentro que los controles y las
# dos cosas se leian corridas. El relleno se DERIVA del margen -- menos el ancho de los
# bordes del propio panel -- para que cambiar uno mueva al otro y no se desincronicen.
_BORDE_IZQ, _BORDE = 4, 1     # los mismos que declara `layout` mas abajo
_RELLENO_IZQ = max(int(fig.layout.margin.l) - _BORDE_IZQ, 0)
_RELLENO_DER = max(int(fig.layout.margin.r) - _BORDE, 0)

PANEL = widgets.VBox(
    [
        _grupo(_titulo('Circuito'), circuito_widget),
        _grupo(_titulo('Ventana'), ventana_widget),
        _grupo(vano_widget,
               widgets.HBox([boton_desmarcar]),
               widgets.HTML(f'<span style="font-size:12px;color:#5b4a48;">Hasta '
                            f'{MAX_VANOS_ANALISIS} vanos a la vez. Tambien puedes marcar y '
                            'desmarcar un vano haciendo clic sobre el en el mapa; sin '
                            'ninguno marcado el simulador toma el circuito completo.'
                            '</span>')),
        _grupo(_titulo('Variables del simulador'), knob_selector_widget,
               AVISO_BLOQUEADOS),
        _grupo(controles_knob_box),
        _grupo(boton_simular),
        _grupo(STATUS),
    ],
    layout=widgets.Layout(
        width='100%', align_items='flex-start',
        padding=f'12px {_RELLENO_DER}px 12px {_RELLENO_IZQ}px', margin='0 0 6px 0',
        border=f'{_BORDE}px solid #e4c4c0',
        border_left=f'{_BORDE_IZQ}px solid rgb(203,24,29)',
    ),
)
PANEL.add_class('panel-v15')

APP = widgets.VBox([ESTILO, PANEL, fig], layout=widgets.Layout(width='100%'))
APP.add_class('app-v15')
# `display` explicito y una sola vez. `add_class` devuelve el propio widget, asi que
# dejarlo como ultima expresion de la celda hacia que Jupyter lo auto-mostrara ADEMAS del
# display de la celda siguiente: el tablero aparecia dos veces.
display(APP)


## Como leerlo

**Los dos mapas.** A la izquierda, la criticidad historica; a la derecha, la simulada.
Van lado a lado y no apilados: la unica comparacion que justifica que haya dos mapas es la
del mismo vano antes y despues de simular, y apilados obligaba a mover la vista de arriba
a abajo para hacerla. Nunca comparten leyenda ni titulo, porque mezclarlos invita a leer
una prediccion como un hecho observado.

Arrancan con el mismo encuadre, pero **al simular el mapa de la derecha se acerca a los
vanos marcados**. Se paga a sabiendas que los dos dejen de mirar la misma geografia: una
vez que el modelo corrio, la pregunta ya no es donde queda el circuito sino que le paso a
ESOS vanos, y buscarlos otra vez dentro del garabato completo es trabajo que el tablero
puede ahorrar. El de la izquierda conserva SIEMPRE el encuadre del circuito, que queda
como la referencia contra la cual se lee el acercamiento; y desmarcar todo devuelve al de
la derecha a esa misma vista.

**Una sola leyenda, horizontal y debajo de los mapas.** Vertical y a la derecha se
llevaba 196 px medidos de ancho -- una columna entera de la fila 3 -- para decir siete
nombres. Y es UNA sola para los dos mapas: usan la misma geometria KMeans de 01.4, asi
que las cuatro clases significan exactamente lo mismo en los dos y repetirlas era decir
dos veces la misma escala.

**El rango de un control sale de valores reales.** `ALTURA` usa 99 como codigo de "sin
dato" -- 327 registros, y el siguiente valor real es 25; un poste de 99 m no existe en
una red de distribucion. Se excluye del rango y del valor inicial, asi que el deslizador
va de 4 a 25 y no de 4 a 99, donde 74 de sus 95 puntos de recorrido caian en un tramo
que ningun vano puede ocupar. Se declara variable por variable en
`vano_controls.VALORES_NO_VALIDOS`: una regla automatica del tipo "los nueves son
relleno" tumbaria el 9 de `LONG_CRUCETA` y el 10 de `VAL_CRIT_APOYO`, que son reales.

**La tabla de variables** dice, de cada control, su rango real y si simularlo significa
algo. No todas las variables del modelo son palancas: unas se mueven con una cuadrilla (la
poda, la puesta a tierra, el conductor), otras son escenarios que nadie controla pero que
son justamente el what-if (el clima, las descargas, el crecimiento de la demanda), otras
describen lo que el vano ES y no algo que se le pueda hacer, y una -- los trafos afectados
EN LA FALLA -- se mide despues del evento que el modelo intenta anticipar, asi que
simularla es circular. El veredicto y su motivo salen del diccionario del propio proyecto
y viven en `simulador_variables.JUICIO_SIMULACION`, con sus pruebas.

La tabla trae la **unidad de medida cuando aplica**: un rango sin unidad no se puede
juzgar -- 25 puede ser una altura razonable o un disparate segun si son metros o pies.
Quedan sin unidad las categoricas, las binarias, los indices como `NR_T` y
`VAL_CRIT_APOYO` -- que son puntajes y no magnitudes -- y `DDT`, cuya descripcion
implica una unidad por area pero no dice cual; antes que estampar una equivocada, la
celda queda vacia.

**Las variables del simulador van en cuatro columnas**: dos para lo que se puede hacer
-- intervencion -- y dos para lo que se quiere anticipar -- escenario. Una lista corrida
de dieciocho casillas obliga a recordar el veredicto de cada una para saber a cual de
las dos preguntas pertenece; en columnas eso lo dice la posicion.

**El panel se alinea con el AREA DE DIBUJO de la figura, no con su borde.** Los dos
ocupan el mismo ancho, pero la figura reserva margen a la izquierda para los rotulos de
los ejes de la primera columna, asi que sus paneles empiezan mas adentro que los
controles. El relleno del panel se deriva de ese margen -- no se escribe a mano -- para
que cambiar uno mueva al otro.

**Las variables refutadas y las de lectura unica ya no aparecen en el panel.** Mientras estuvieran ahi, el
tablero las ofrecia como equivalentes a la poda o a la puesta a tierra. Las "Limitado"
salen por el mismo motivo: hay UNA lectura bajo la cual se interpretan -- adelantar una
fecha equivale a reponer el activo -- y un deslizador no puede transmitir esa condicion,
quien lo mueve ve el numero y no el motivo. Quitarlas NO las
saca de la simulacion: un override solo se escribe si se fija, asi que entran al modelo
con el valor OBSERVADO de cada vano, que es exactamente lo que corresponde -- lo unico
que se pierde es poder moverlas. El panel las nombra debajo de la lista de variables, o
acortarse sin explicacion se leeria como que faltan.

**Negro = sin evento**, en los dos mapas. Un vano sin eventos en la ventana no tiene clase,
y la ausencia no es el grupo mas bajo. En el mapa simulado el negro cubre ademas lo que
quedo fuera de la seleccion.

**Se analizan hasta 5 vanos a la vez.** Cada vano marcado recibe su propia COLUMNA de
controles abajo del panel, rotulada con el vano que gobierna, y cada control abre en el
valor ACTUAL de esa variable para ESE vano en la ventana activa -- la mediana de sus
instancias, o la moda si la variable es categorica. Eso es lo que permite preguntar "que
pasa si podo SOLO este": los demas vanos quedan exactamente como estaban, no en un valor
por defecto. Al llegar a los cinco las casillas restantes se deshabilitan solas, y el clic
en el mapa respeta el mismo tope. Sin ningun vano marcado el grano vuelve a ser el circuito
completo y hay una sola columna, cuyos valores se aplican a todo. Un vano marcado SIN
eventos en la ventana activa no recibe columna -- no hay valor actual desde donde arrancar
-- y el pie de la rejilla lo nombra en vez de dejarlo desaparecer en silencio.

**Marcar un vano** se hace con su casilla o tocandolo en el mapa de la izquierda: las dos
vias son el mismo estado, porque el clic alterna la casilla. Solo el mapa base acepta clic;
el de la derecha es la salida del modelo, no un control, y marcar desde ahi mezclaria "lo
que elegi" con "lo que el modelo predijo" sobre la misma superficie. Un vano marcado se dibuja con el
color de SU clase sobre un halo blanco -- no con un color plano de "seleccionado", que
congelaria lo que se ve cuando la ventana cambia la clase por debajo.

**La caja amarilla** encierra cada vano marcado: es su rectangulo envolvente, translucido,
y esta para responder *cual estoy estudiando* en un circuito de cientos de tramos, donde
un trazo un poco mas ancho ya no basta. Va DEBAJO de las lineas del mapa, asi que no tapa
el color de clase del vano ni se come el clic. Sale de la geometria y no de los datos de
la ventana: **sigue puesta al mover el deslizador de ventana**, incluso sobre un vano que
en esa ventana no registro un solo evento. Se apaga de una sola forma -- desmarcando el
vano, con su casilla o volviendo a tocarlo en el mapa.

**En el mapa simulado el mismo recuadro cambia de pregunta**: alli el vano ya esta
identificado a la izquierda, asi que el color dice QUE LE PASO. Verde claro si bajo de
grupo de criticidad, amarillo si se quedo en el mismo, rojo si subio. El amarillo es
exactamente el del mapa base porque "no cambio" es justo el estado en que los dos mapas
dicen lo mismo. Son tres capas y no una porque una capa del mapa pinta con un solo color.
Un vano marcado que la simulacion no puntuo -- sin celda en la ventana activa, o marcado
DESPUES de presionar "Simular" -- no recibe recuadro: no tiene grupo base ni simulado, y
pintarlo de amarillo afirmaria que no cambio, que es justo lo que nadie midio.

**La etiqueta del mapa simulado trae los DOS grupos**, el base y el simulado, sobre el
mismo vano. Sin eso, saber si un vano mejoro obliga a cruzar al mapa de al lado y
acordarse del color. El grupo base viaja por punto y no en la plantilla de la traza porque
dentro de una traza -- que es una clase SIMULADA -- el grupo base cambia de vano a vano.

**"Simular" es el unico disparador** y produce las tres salidas en el mismo trabajo,
aplicando solo las variables elegidas en el panel. Cada variable aparece como un control
-- deslizador si es numerica, lista si es categorica -- y una familia climatica
(precipitacion, temperatura, rafaga y viento) mueve sus 12 rezagos horarios de una vez.
El mapa simulado **no existe hasta que se presiona**: antes solo muestra el aviso, porque
un mapa pintado de "aun no simulado" ocupa el mismo lugar y tiene la misma forma que un
resultado.

**Cambiar circuito o ventana descarta la ultima simulacion.** La fila 2, el grafo y la
importancia se vacian: mostrar la corrida de otra seleccion es la misma confusion que
todo lo anterior evita.

**El "Grafo de relevancia"** (fila 4, columnas 1-2) es el grafo experto tal como lo usa la
seleccion: `media_vanos(compuerta) x peso_fijo` por arista, en disposicion circular. Cada
nodo lleva su variable y el color de su modo -- `climaticos` o `estructurales` --, paleta
deliberadamente ajena a la de los grupos KMeans. Se **anula** cuando las compuertas no
varian entre los vanos, lo que incluye por construccion cualquier seleccion de menos de 3
vanos: dibujarlo igual seria presentar el grafo experto fijo como si lo hubiera estimado
esta seleccion.

**"UITI acumulado y eventos por ventana"** (fila 3, columnas 1-2) es la serie de tiempo de
los vanos elegidos, y **solo aparece al elegirlos**: sin ninguno marcado queda vacia a
proposito, por el mismo motivo que los violines de 01.4 -- una serie sobre el circuito
entero y una sobre tres vanos se dibujan igual y no miden lo mismo, asi que caer al
circuito cambiaria el sujeto del panel en silencio.

Conviven DOS codigos de color sobre el mismo punto, separados por canal para que los dos
se lean a la vez: la **linea y el anillo** llevan el color de identidad del vano -- dicen
de QUE vano es la serie -- y el **relleno del punto** lleva el color del grupo de riesgo
en que cayo ese vano en ESA ventana, con la paleta del mapa. Por eso un mismo vano cambia
de relleno a lo largo de su serie: medido, uno pasa de Bajo a Medio-Alto y vuelve a Bajo
en cuatro ventanas. Un relleno gris es una ventana sin celda, que no tiene grupo --
distinto de caer en el mas bajo.

El eje x lleva la **fecha en que empieza cada ventana**, no su etiqueta `V1`, `V2`: un
rotulo `V7` obliga a ir a buscar a que periodo corresponde cada vez que se mira el panel.
Cada celda (vano, ventana) es UNA fila ya agregada, asi que cada punto tiene exactamente
una clase: a este grano no hay un conjunto de etiquetas del que tomar la moda.
Doble eje porque UITI y eventos viven en escalas muy distintas. Una ventana sin celda va
como hueco y no en cero -- no es "no hubo UITI", es "no hubo medicion" -- y la linea se
corta ahi. El punto de la ventana vigente se dibuja al triple y viaja con el deslizador.

**"Top 10 de variables por vano"** (fila 3, columnas 3-4) es un grupo de barras por vano,
con el nombre de la variable escrito dentro de la barra. Responde la pregunta que sostiene
una orden de trabajo: cual mueve a ESTE vano. No es SHAP: es el mismo barrido min-max del
resto del cuaderno, sobre el mismo modelo y la misma unidad. Los cinco vanos cuestan las
MISMAS pasadas que uno, porque cada pasada ya devuelve un u-hat por bolsa -- y pasar de
cinco a diez variables no cuesta ninguna pasada mas, porque el barrido ya las puntuo
todas: el top solo decide cuantas se dibujan.

El rotulo dentro de la barra se elige **barra por barra segun lo que mida**: el nombre
resumido si cabe, sus iniciales si no, y nada antes que un texto cortado que se monte
sobre la barra vecina. Plotly no sabe hacer esa cascada -- o escribe el texto entero o lo
esconde --, asi que la decide el cuaderno con el largo de cada barra en pixeles. **El
nombre completo esta siempre en la etiqueta del mouse**, que es donde se resuelve la duda.
El color de la barra codifica la POSICION en el ranking y nada mas: es una rampa de un
solo color, deliberadamente ajena a la paleta de los grupos, para que una barra no se lea
como un grupo de criticidad.

**"UITI de la seleccion: base contra simulado"** (fila 4, columnas 3-4) mide la misma
cantidad dos veces sobre los mismos vanos. Es la lectura que cierra el tablero: si la
simulacion no mueve la distribucion, no la movio, y eso se ve sin comparar dos mapas tramo
a tramo. Con pocos vanos un violin es casi una linea, por eso van tambien los puntos: con
tres o cuatro datos lo honesto es mostrarlos, no dibujar una densidad que no existe.


## La matematica de lo que hace "Simular"

Todo lo de abajo describe UNA pulsacion del boton sobre la seleccion activa
$(c, w, M)$: circuito, ventana y conjunto de vanos marcados.

### 1. Las bolsas de la seleccion

La unidad de prediccion no es el evento: es la **bolsa**, la celda
$(\text{circuito}, \text{vano}, \text{ventana})$. Es la misma unidad en la que 04 define
la criticidad, y por eso el mapa simulado se puede comparar con el historico.

$$\mathcal{B}(c,w,M)=\{\,b=(c,v,w)\;:\;v\in M\,\},\qquad M=\varnothing\;\Rightarrow\;M:=V(c,w)$$

donde $V(c,w)$ son todos los vanos del circuito con al menos un evento en esa ventana:
sin vanos marcados el grano es el circuito completo, no un panel vacio.

Cada bolsa $b$ agrupa sus instancias $I_b$ -- las filas de evento de ese vano dentro de
esa ventana -- y trae dos cosas que **no se predicen nunca**:

$$n_b=|I_b|\quad(\text{eventos OBSERVADOS}),\qquad x_i\in\mathbb{R}^{p},\;p=80$$

Las $p=80$ columnas son 22 estructurales + 48 rezagos de clima + `COD_CAUSA` y sus 9
indicadores. Conviene no confundir esa cuenta con la **particion por modalidades** que usa
el modelo, que no es la misma: `climaticos` son las 50 columnas de los 48 rezagos **mas
`DDT` y `NR_T`** (descargas y nivel de tormenta son clima, aunque viajen como columnas
estaticas), y `estructurales` son las 30 restantes -- las otras 20 estructurales mas
`COD_CAUSA` y sus 9 indicadores. El almacenamiento es CSR (`offsets`, `counts`), no una matriz rellenada:
el 52,7% de las bolsas son de un solo evento y el maximo es 46, asi que rellenar
desperdiciaria mas de 40x en la mitad de los datos. Al seleccionar, el indice de bolsa se
**renumera** desde 0, porque el modelo toma `n_bags = max(instance_bag)+1` y los ids
originales reservarian una bolsa vacia por cada celda no seleccionada.

**Los controles del simulador** actuan sobre las instancias, no sobre las bolsas. Un
control $\kappa$ gobierna un conjunto de columnas $F(\kappa)$ -- una sola para una
variable estructural, las 12 de una familia climatica -- y aplicarlo es

$$x_{i,j}\;\leftarrow\;\phi_j(\text{valor}),\qquad \forall\, i\in\textstyle\bigcup_b I_b,\;\forall\, j\in F(\kappa)$$

con $\phi_j$ la coercion a espacio de modelo (categoria por su codificador, fecha a
epoch, NaN a su centinela). No hay escalador despues: la matriz de instancias del MIL es
espacio crudo. **$n_b$ jamas se toca**: es un eje del espacio que define la clase, y
moverlo desplazaria al vano por una dimension que el modelo no predice.

### 2. La prediccion de clase de cada vano

El modelo hace **dos pasadas** sobre el mismo codificador. La primera existe solo para
producir las compuertas del grafo.

**(a) Codificacion y atencion.** Cada instancia se codifica por modalidad
(30 estructurales, 50 climaticas, segun la particion de arriba) y se concatena en
$z_i^{(1)}$. La bolsa se resume con
atencion tipo Ilse, normalizada dentro de la bolsa:

$$e_i=\mathbf{w}^{\top}\tanh(V z_i^{(1)}),\qquad
a_i=\frac{\exp(e_i)}{\sum_{i'\in I_b}\exp(e_{i'})},\qquad
z_b^{(1)}=\sum_{i\in I_b}a_i\,z_i^{(1)}$$

Esto es **invariante a la cardinalidad por construccion**: duplicar cada instancia de una
bolsa no cambia ningun $e_i$, el denominador se duplica, cada copia recibe $a_i/2$ y la
suma queda igual.

**(b) Compuertas del grafo experto.** Un decodificador lee el resumen de la bolsa y
produce una compuerta por arista:

$$g_b=2\,\sigma(W_g\,z_b^{(1)})\;\in\;(0,2)^{E},\qquad E=64$$

Se inicializa en cero, de modo que $g_b=\mathbf{1}$ al arrancar y el grafo aprendido
**parte exactamente del grafo experto fijo**.

**(c) Propagacion.** El grafo experto es una adyacencia fija $W\in\mathbb{R}^{80\times 80}$
con soporte en $E$ aristas. Cada instancia recibe, en la columna destino de cada arista:

$$x'_{i,j}\;=\;x_{i,j}\;+\;\alpha\!\!\sum_{e\,:\,\mathrm{dst}(e)=j}\!\! g_{b(i),e}\;w_e\;x_{i,\mathrm{src}(e)},
\qquad \alpha=0{,}2$$

Una columna que no es destino de ninguna arista queda intacta, exactamente.

**(d) Segunda pasada y fusion FiLM.** El MISMO codificador procesa $x'$, se vuelve a
agrupar con la MISMA atencion y las dos modalidades se fusionan modulando la
estructural con la climatica (`film_modulated_modality = estructurales`, del artefacto):

$$\hat z_b=z_b^{\text{est}}\odot\bigl(1+\gamma(z_b^{\text{clim}})\bigr)+\beta(z_b^{\text{clim}}),
\qquad p_b=h(\hat z_b),\qquad \hat u_b=\mathrm{expm1}(p_b)$$

La concatenacion es aditiva entre modalidades y no puede representar un producto entre una
variable estructural y una climatica; FiLM hace que el clima **reescale** lo estructural,
que es tambien la afirmacion de dominio: una rafaga pesa mas sobre un apoyo alto, viejo y
degradado. El modelo aprende en $\log(1+u)$ y se devuelve a UITI con `expm1`.

**(e) Clase.** Con el UITI predicho y los eventos observados, la clase sale de la
geometria KMeans de 04 -- **la misma que pinta el mapa base**, verificada al cargar el
modelo. En el espacio canonico `2` el eje de eventos es lineal y el de UITI logaritmico:

$$\zeta_b=\left(\frac{n_b-\mu_0}{s_0},\;\frac{\log_{10}\max(\hat u_b,\varepsilon)-\mu_1}{s_1}\right),
\qquad \hat k_b=\arg\min_{k\in\{0,1,2,3\}}\lVert \zeta_b-c_k\rVert^2$$

El mapa pinta $\hat k_b$ con la paleta de los cuatro grupos; lo que no tiene bolsa en la
ventana, o quedo fuera de la seleccion, va en negro. El simulador corre esto **dos veces**
-- sin y con los controles aplicados -- y $\Delta_b=\hat k_b^{\text{sim}}-\hat k_b^{\text{base}}$
es cuantos vanos cambian de clase.

### 3. La relevancia de variables, vano por vano

Es un barrido min-max, **no SHAP**. Se mide sobre las mismas bolsas del mapa, con el
riesgo ordinal esperado de **cada bolsa** -- la distribucion suave sobre las cuatro clases,
sin promediar entre vanos:

$$P_{b,k}=\frac{\exp(-\lVert\zeta_b-c_k\rVert^2/\tau)}{\sum_{k'}\exp(-\lVert\zeta_b-c_{k'}\rVert^2/\tau)},
\qquad
R_b(X)=\sum_{k}k\,P_{b,k}\;\in[0,3]$$

No promediar es lo unico que hace falta para que **un solo barrido sirva para los cinco
vanos**: cada pasada del modelo ya produce un $\hat u_b$ por bolsa, y promediarlo de
inmediato tiraba esa informacion y obligaba a repetir el barrido vano por vano.

Para cada control **numerico** $\kappa$, con su rango observado $[m_\kappa,M_\kappa]$ en
todo el dataset, se lleva a sus dos extremos todas las columnas $F(\kappa)$ a la vez:

$$\Delta^{-}_{b,\kappa}=R_b\!\left(X^{\kappa\to m_\kappa}\right)-R_b(X),\qquad
\Delta^{+}_{b,\kappa}=R_b\!\left(X^{\kappa\to M_\kappa}\right)-R_b(X),\qquad
s_{b,\kappa}=\max\!\left(|\Delta^{-}_{b,\kappa}|,\,|\Delta^{+}_{b,\kappa}|\right)$$

El panel dibuja, por cada vano, sus **diez** $s_{b,\kappa}$ mas grandes, ordenados. La
magnitud va **cruda** y no normalizada: la pregunta que sostiene una orden de trabajo es
cuanto mueve una variable a ESTE vano, y un softmax por vano haria que cinco rankings con
magnitudes muy distintas se dibujaran identicos. La escala del eje es comun a los cinco
grupos, asi que dos barras de la misma altura significan lo mismo en cualquier grupo.

Los controles **categoricos y constantes se omiten**: no tienen minimo/maximo numerico, e
inventarles un rango seria puntuar un escenario que nadie pidio. Los que el panel no
ofrece -- refutados y de lectura unica -- tampoco entran al barrido: no se puede rankear
por relevancia lo que no se deja mover.

### 4. El grafo inferido

Las compuertas de la parte (b) son lo unico del grafo que depende de la seleccion. Se
juntan en una matriz $G\in\mathbb{R}^{|\mathcal{B}|\times E}$, con $G_{b,e}=g_{b,e}$.

Antes de reconstruir nada se mide si esas compuertas **varian** entre vanos. Con
$\tilde G$ la matriz centrada por columnas y $\sigma_1,\dots$ sus valores singulares:

$$\mathrm{var}=\frac{1}{E}\sum_e \mathrm{Var}_b(G_{b,e}),\qquad
\mathrm{rank}_{\text{ef}}=\frac{\bigl(\sum_r\sigma_r^2\bigr)^2}{\sum_r\sigma_r^4},\qquad
\text{colapso}\iff \max_e \mathrm{std}_b(G_{b,e})<10^{-6}\;\lor\;\mathrm{rank}_{\text{ef}}\le 1$$

El rango efectivo es el cociente de participacion: vale $\approx 1$ cuando toda la
variacion vive en una sola direccion, es decir, cuando todos los vanos estan compuertados
igual. Un colapso **anula** el grafo y el panel lo dice, en vez de dibujar el grafo
experto fijo como si lo hubiera estimado esta seleccion. De ahi sale un limite duro:
con $|\mathcal{B}|<3$ la matriz centrada tiene rango 1 por construccion, asi que **menos
de 3 vanos nunca producen grafo**.

Si no hay colapso, el peso reconstruido de cada arista es el peso experto fijo tal como lo
usa esta familia de vanos:

$$\bar g_e=\frac{1}{|\mathcal{B}|}\sum_{b}G_{b,e},\qquad
A_{\mathrm{src}(e),\,\mathrm{dst}(e)}=\bar g_e\cdot w_e,\qquad A_{ij}=0 \text{ fuera del soporte}$$

El panel lo dibuja en disposicion circular sobre las variables que participan de al menos
una arista, con $A_{ij}$ en un marcador sobre el punto medio de cada arista.

### Presupuesto de una pulsacion

| Paso | Pasadas de bolsas |
|---|---|
| Mapa simulado (base + simulado) | 2 |
| Compuertas para el grafo | 1 |
| Relevancia (base compartida + min/max por control) | $1+2K$ |

$K$ son los controles **numericos que el panel ofrece**, y de ahi sale la propiedad que
sostiene el panel: subir el top de cinco a diez variables **no cuesta una sola pasada
mas**. El barrido ya calculo $s_{b,\kappa}$ para todos los controles y para todas las
bolsas; el top solo decide cuantos de esos numeros se dibujan.

Medido sobre el modelo real: 0,06 s para la seleccion tipica del panel. Por eso alcanza
con recorrer las bolsas una vez por pulsacion y no hace falta cache en disco.

## De donde sale el UITI de los violines

Los dos violines de la fila 4 miden **la misma cantidad dos veces sobre los mismos vanos**.
Vale la pena decir con precision cual, porque el nombre "UITI acumulado" tambien es el de
una columna del historico y **no es esa** la que se dibuja aqui.

### La unidad es la bolsa, no el evento

Cada punto de un violin es **una bolsa**: la celda $(\text{vano}, \text{ventana})$ de la
seleccion activa. Con la ventana fija y hasta cinco vanos marcados, un violin tiene como
maximo cinco puntos -- por eso van dibujados uno por uno (`points='all'`) y no solo como
densidad: con tres o cuatro datos, lo honesto es mostrarlos.

### Los dos numeros

Las bolsas de la seleccion aportan su matriz de instancias $X$ -- una fila por evento --
y su mapa instancia $\to$ bolsa. De ahi salen los dos vectores, con **una pasada del
modelo cada uno**:

$$\hat u^{\text{base}}_b=f_\theta\bigl(X,\;\text{bolsa}\bigr)_b,
\qquad
\hat u^{\text{sim}}_b=f_\theta\bigl(X',\;\text{bolsa}\bigr)_b$$

donde $f_\theta$ es el modelo MIL del cuaderno 05 -- el mismo que pinta los dos mapas -- y
$X'$ es **exactamente $X$** con las columnas de los controles fijados sobreescritas, cada
vano en sus propias filas. Todo lo que no se toco en el panel entra a $X'$ con su valor
**observado**: un control que no se fija no escribe nada.

### Lo que hay que tener claro al leerlos

- **El violin "Base" tambien es una prediccion**, no el historico. Es el modelo puntuando
  los mismos vanos con sus valores observados. Se compara prediccion contra prediccion a
  proposito: asi lo unico que separa a los dos violines es lo que se movio en el panel.
  Contra el UITI medido se estaria midiendo otra cosa -- el efecto de la simulacion mas el
  error del modelo, mezclados y sin forma de separarlos.
- **La escala es $\hat u$, el UITI acumulado predicho de la bolsa**, la misma cantidad que
  entra al eje vertical de la geometria KMeans -- por eso mover un violin y ver cambiar de
  grupo un vano en el mapa son la misma cosa vista dos veces.
- **$n_b$, los eventos observados, jamas se simula**. Es el otro eje del espacio que define
  la clase. De ahi que toda la diferencia entre los dos violines venga de $\hat u$ y de
  nada mas.
- **Si los dos violines se superponen, la simulacion no movio nada.** Es la lectura que
  cierra el tablero y se hace de un vistazo, sin comparar dos mapas tramo a tramo.
- Los violines describen a los **vanos marcados**. Sin ninguno marcado el grano es el
  circuito completo, y entonces cada punto es un vano del circuito con eventos en la
  ventana: mas puntos y otra pregunta.

### Lo que cuesta

**Dos pasadas de bolsas en total**, nunca una por vano: los valores de cada vano se
escriben en la MISMA matriz y se puntuan juntos. Es la misma corrida que produce el mapa
simulado -- los violines no vuelven a llamar al modelo, leen la tabla que ya devolvio.
